# Lab 3: 量子優位性のためのあなたの新しいツール

Qiskit Global Summer School 2026 の Lab 3 へようこそ！

このラボでは、__[Samplomatic](https://github.com/Qiskit/samplomatic)__ の使い方を学びます。これはエラー緩和のプロセスを高度に制御できるようにするために導入されたフレームワークです。これは量子優位性に向けた私たちの旅における重要な一歩です。

- __第1章__ では、Samplomatic を _使わずに_ Qiskit Runtime で可能なエラー緩和のプロセスを扱います。_層をドレスする（dress a layer）_ とはどういうことか、そしてパウリがどのようにクリフォードゲートを伝播していくかを学びます！これらは第2章のための本質的な概念です。

- __第2章__ では、Samplomatic を使うために知っておくべきことをすべて学びます。回路をボックス化する方法、層ごとにノイズを学習する方法、そして _Executor_ プリミティブでジョブを実行する方法です。

- 最後に __第3章__ では、Samplomatic を使う2つの Qiskit アドオンを紹介します。[Propagated Noise Absorption](https://qiskit.github.io/qiskit-addon-pna/)（PNA）と [Shaded Lightcones](https://qiskit.github.io/qiskit-addon-slc/)（SLC）です。

__使用量の情報:__ _Heron デバイスでの合計 QPU 使用量の見積もりは 1 分 30 秒です。この使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。_

このラボは QPU ハードウェア上での実行が必要です。高度なエラー緩和を行うためにデバイスのノイズを学習するので、すべてのノイズ学習（noise Learner）ジョブでは実機ハードウェアでの実行が必要です。これらのジョブをシミュレーターで実行することはできません！

__Windows ユーザーへの注意:__
第3章で使う shaded lightcone アドオンは、Python ベースの化学シミュレーションライブラリーである `pyscf` パッケージを使用しますが、これは Windows でネイティブにはサポートされていません。Windows ユーザーには、qBraid のようなクラウドベースのサービスを使うか、Windows Subsystem for Linux を使って Python 環境を作成することをおすすめします。

## セットアップとインポート
下の `pip install` のコメントを外してセルを実行し、必要なライブラリーをインストールしてください。その後、カーネルを再起動するのを忘れないでください！

In [ ]:
# 必要なパッケージをインストールする
#!pip install -q -U qiskit qiskit-ibm-runtime
#!pip install -q matplotlib numpy ipython
#!pip install -q samplomatic
#!pip install -q qiskit-addon-utils qiskit-addon-pna qiskit-addon-slc
#!pip install --upgrade qc-grader
#!pip install plotly
#!pip install pylatexenc
#!pip install nbformat

In [ ]:
### 科学計算
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Patch

### IPython
from IPython.display import display

### Qiskit 
from qiskit import QuantumCircuit
from qiskit.quantum_info import Pauli, SparsePauliOp
from qiskit.transpiler import generate_preset_pass_manager

### Qiskit IBM Runtime
from qiskit_ibm_runtime import QiskitRuntimeService, Executor, QuantumProgram
from qiskit_ibm_runtime.options import EstimatorOptions
from qiskit_ibm_runtime.options_models.noise_learner_v3_options import NoiseLearnerV3Options
from qiskit_ibm_runtime.noise_learner_v3 import NoiseLearnerV3

### Samplomatic
import samplomatic
from samplomatic import Twirl, InjectNoise, ChangeBasis, build
from samplomatic.transpiler import generate_boxing_pass_manager
from samplomatic.utils import find_unique_box_instructions, get_annotation

### Qiskit アドオン
from qiskit_addon_utils.exp_vals.measurement_bases import get_measurement_bases
from qiskit_addon_utils.exp_vals.expectation_values import executor_expectation_values
from qiskit_addon_utils.noise_management import trex_factors, gamma_from_noisy_boxes
from qiskit_addon_pna import generate_noise_mitigating_observable
from qiskit_addon_slc.bounds import compute_backward_bounds, compute_forward_bounds, compute_local_scales, merge_bounds
from qiskit_addon_slc.utils import generate_noise_model_paulis, map_modifier_ref_to_ref
from qiskit_addon_slc.visualization import draw_shaded_lightcone


# Grader のインポート
from qc_grader.challenges.qgss_2026 import (
    grade_lab3_ex1,
    grade_lab3_ex2,
    grade_lab3_ex3,
    grade_lab3_ex4,
    grade_lab3_ex5,
)

# 進捗を確認するための grader 関数
from qc_grader.challenges.qgss_2026 import check_progress

#### バックエンドを定義する:

このラボでは __Heron デバイス__ を使って作業します。上で述べたように、バックエンド上で実行するノイズ学習ジョブはシミュレーターでは実行できません。Heron デバイスに絞ることで、Open Plan アカウントで作業する参加者もこのラボに参加できます。

このチュートリアルを通して _同じバックエンド_ を使うことをおすすめします。バックエンドのノイズを学習し、そのノイズモデルを使って高度なエラー緩和の手法を実装します。したがって、途中でバックエンドを変えると、学習したノイズモデルは新しいバックエンドには適用できなくなります。

注意: バックエンドへの接続に問題がある場合は、Lab 0 に戻って IBM Quantum Platform アカウントが正しく設定されているか確認してください。

In [ ]:
service = QiskitRuntimeService()

# バックエンドを定義する: 
service = QiskitRuntimeService()
backend = service.least_busy(
    operational=True,
    filters=lambda b: b.processor_type.get('family') == 'Heron'
)

print(f"Backend     : {backend.name}")
print(f"# qubits    : {backend.num_qubits}")
print(f"Basis gates : {backend.basis_gates}")

# optimization_level=0 でトランスパイルするための ISA パスマネージャーを作る
isa_pm = generate_preset_pass_manager(backend=backend, optimization_level=0)

## 第0章 — 今日の量子ハードウェアにおけるエラー

今日の量子ハードウェアはノイズが多いです。すべてのゲート、すべてのアイドル時間、すべての測定が小さなエラーをもたらし、実用規模（100 量子ビット以上の回路）では、それらのエラーがもはや無視できない程度にまで蓄積します。完全な誤り耐性が実現するまで、これらのデバイスから *有用な* 答えを得る唯一の方法は、そのノイズを管理することです。すなわち、抑制できるところでは抑制し、できないところでは特性を評価（characterize）し、結果を読み出すときにはそのバイアスを緩和することです。したがってノイズへの対処は、今日のデバイスから有用な結果を得るための中心的な要素であり、そのための技術は活発に開発が進む分野です。

ノイズを扱う技術は、おおまかに4つのカテゴリーに分けられます。これらは、*いつ* 作用するか、*何を* コストとするか、そしてエラーを修正するのか、フラグを立てるのか、それとも事後に統計量だけを補正するのか、という点で異なります。そのため、このラボの中心となる新しいツールを紹介する前に、この4つすべてを見ておく価値があります。

- **誤り訂正（EC）** は、論理情報を冗長にエンコードし、シンドロームを繰り返し抽出します。誤り耐性のある形で実装されれば、論理エラーを物理エラー率よりも低く抑えることができます。完全な QEC は、その量子ビットと時間のオーバーヘッドのため、近未来のデバイスでは実現困難です。

- **エラー検出（ED）** は、破損している可能性が高い実行にフラグを立て、事後選択（post-selection）によってそれらを破棄します。これによって、より小さいがよりクリーンなサンプルを保持します。このフラグは、検出のみの QEC 符号、すなわち距離（distance）によってエラーを知らせるものの、それを特定して元に戻すのに十分な情報は与えない符号から得られる場合があります。その場合、ED は誤り訂正の極限的なケースになります。ED によって、階層的な方式における上位の符号は、フラグの立ったエラーを *消失（erasure）* として扱うことができます（既知の場所は、未知の場所よりもはるかに訂正しやすいのです）。同じフラグは、より安価で符号を使わないチェックからも得られます。たとえば、理想的な回路が保つべき対称性（保存されるパリティや粒子数など）を検証し、それに違反したショットを棄却するようなものです。いずれにせよ、ED は使用可能なサンプルの減少と引き換えに、誤り耐性のある EC よりもはるかに低い実装オーバーヘッドで済みます。

- **エラー緩和（EM）** は個々のショットをまったく修正しません。多数のノイズを含む実行にわたって統計量にバイアスをかけたり再重み付けしたりすることで、期待値（や分布）の *推定値* を理想的なものに近づけます（[Cai et al., *Quantum error mitigation*, Rev. Mod. Phys. 95, 2023](https://journals.aps.org/rmp/abstract/10.1103/RevModPhys.95.045005)）。これは今日の実用規模の実験が動作する領域であり、このラボの焦点です。

- **エラー抑制（ES）** は、実行中に作用してノイズそのものを整形（reshape）します。例としては、アイドル状態の量子ビットにパルスを挿入してコヒーレントエラーを打ち消す動的デカップリング（DD）や、ゲートのドレッシングをランダム化して残留エラーを確率的なパウリチャネルにするパウリトワリング（PT）があります（[Wallman & Emerson, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)）。

この4つのうち2つは、Qiskit Runtime プリミティブを実行したことがあれば、気づいていたかどうかにかかわらず、すでに使ったことがあります。`Estimator` が実装しているのはエラー抑制とエラー緩和です（誤り訂正と検出はプリミティブが適用してくれるものではありません）。デフォルトのオプションでも、`Estimator` はすでに系統的な読み出しバイアスを補正します（Twirled Readout Error eXtinction、TREX）。さらに進んで、1.1.2 節で見るように `EstimatorOptions` を使って追加のエラー抑制やエラー緩和の手法を自分で有効にすることもできます。これらのオプションすべてに共通するのは、あなたの回路に対して単一の _回路全体（circuit-wide）_ のポリシーとして作用し、すべての実行に一定の種類のオーバーヘッドを加えるという点です。

このラボは、Qiskit Runtime と並んで使えるツール（_Samplomatic_）を紹介します。これによって _回路全体_ の設定を超え、層ごと・オブザーバブルごとにノイズを管理できるようになります。第1章では、なじみのある `EstimatorOptions` が何をするのか、そして何をコストとするのかの概要から始めます。

## 第1章 — Runtime プロダクトとしてのエラー緩和、そしてそのコストの背後にある理論

この章では、なじみのある Runtime オプションを取り上げ、その中身を開いていきます。
- 1.1 節では、`Estimator` が提供するエラー抑制とエラー緩和の手法を一つずつ見ていき、それぞれを手作業で設定します。

- 1.2 節では、それらのスイッチの *1つ下* のレベルに降りて、これらの技術が拠って立つ最も基本的な原理である小さな代数 — *ドレッシング* と *交換関係（commutation）* — を扱います。

- 1.3 節では、これらの回路全体のスイッチが *できない* ことは何か、そしてリソースオーバーヘッドの観点で何をコストとするかを問いかけて章を締めくくります。これが、このラボの残りの部分の土台となる、よりきめ細かいボックス単位のツールへの動機付けになります。

## 1.1  クラウドサービスとしてのエラー緩和

はじめに述べたように、Qiskit Runtime はエラー緩和をプロダクトとして公開しています。すなわち、オンにする一連のオプションであり、あなたの回路に単一の回路全体のポリシーとして適用されます。この節では、`EstimatorV2` プリミティブ上のそれらのオプションをカタログ化します。各オプションが何をするか、そしてどの `resilience_level` プリセットがそれをオンにするか（1.1.1 節）を示し、その後、各手法の設定と挙動を詳しく見ます（1.1.2 節）。最後に演習1があり、そこでは5つの手法それぞれを個別の `EstimatorOptions` に設定し、各手法の効果を分離できるようにします。これらの手法が拠って立つ代数は 1.2 節で扱います。

### 1.1.1  Runtime プリミティブのオプション

`EstimatorV2` プリミティブは、これらのエラー抑制とエラー緩和の技術を単一の `EstimatorOptions` オブジェクトの背後にまとめています。公式のカタログは [Error mitigation and suppression techniques](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques) です。`resilience_level` オプションは、固定された一連のオプションを一度にオンにする便利なプリセットです。下の表は、各技術、その役割、そしてどのプリセットレベルがそれを有効にするかを示しています。正確な属性のパスは 1.1.2 節で紹介します。

| 技術 | 役割 | 一行での説明 | 有効化 |
|---|---|---|---|
| **DD**（動的デカップリング） | 抑制 | アイドル状態の量子ビットにパルス列を挿入し、量子ビットがアイドルの間に蓄積するコヒーレントエラーを打ち消す | 明示的に有効化 |
| **PT**（パウリトワリング、ゲート） | 抑制 | 2量子ビットゲートノイズのコヒーレントな部分を実質的に確率的なパウリチャネルに整形する。これにより下流の手法（PEA、PEC）がパウリチャネルノイズを仮定できるようになる | レベル2 |
| **TREX**（Twirled readout error extinction） | 緩和 | ランダムな X ゲートで測定をトワリングし、読み出しエラーの転送行列を対角化してリスケーリングで反転できるようにする | レベル1（デフォルト） |
| **ZNE**（ゼロノイズ外挿） | 緩和 | いくつかの制御された係数でノイズを増幅し（例: ゲートフォールディングによって）、期待値をゼロノイズ極限へ外挿する（不偏性は保証されない） | レベル2 |
| **PEA**（確率的エラー増幅） | 緩和 | ZNE のゲートフォールディングによる増幅を、*学習された* パウリ・リンドブラッドノイズモデルによって駆動される増幅に置き換え、より正確なノイズスケーリングを行う（外挿ステップは引き続き ZNE のものを使う） | 明示的に設定（ZNE とともに） |
| **PEC**（確率的エラーキャンセル） | 緩和 | アンサンブル平均が学習されたノイズチャネルを反転する反ノイズ（anti-noise）回路をサンプリングし、サンプリングオーバーヘッド $\gamma^2$ を代償として不偏な推定値を与える | 明示的に設定 |

デフォルトでは Runtime は **resilience level=1** で動作し、これは TREX のみを有効にします。各レベルが何をオンにするかについては [Configure error mitigation](https://docs.quantum.ibm.com/guides/configure-error-mitigation) ガイドを参照してください。次の節では、各技術のオプションと挙動を詳しく見ます。

### 1.1.2  各手法の詳細

この節では、1.1.1 節の表の各技術を展開します。すなわち、何をするか、それを制御するオプション、そしてどんなときに役立つかです。図と完全な説明は公式の [Error mitigation and suppression techniques](https://docs.quantum.ibm.com/guides/error-mitigation-and-suppression-techniques) ガイドからのものです。

#### 動的デカップリング（DD） — *抑制*
アイドル状態の量子ビットは、他の量子ビットの処理が終わるのを待つ間にコヒーレントエラーを拾います。DD は、それらのアイドル状態の量子ビットに、掛け合わせると恒等演算になるパルス列を挿入します。したがって意図した計算は変わらないまま、パルスがゆっくりしたコヒーレントなドリフトを平均化して消し去ります。回路にアイドルの隙間があるときに最も役立ちます。密に詰まった回路では、パルス自体が不完全なため、ほとんど効果がないか、かえって悪化させることもあります。

![動的デカップリング: 量子ビットのアイドル期間中に挿入される XX パルス列（IBM Quantum docs）。](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/dd.avif)

- `dynamical_decoupling.enable` — DD をオンにする（デフォルトはオフ）。
- `dynamical_decoupling.sequence_type` — どのパルス列を使うか。デフォルトは `"XX"`、その他に `"XpXm"`、`"XY4"` など。

#### パウリトワリング（PT） — *抑制*
トワリング（ランダム化コンパイル）は、選んだゲートを、理想的な演算が変わらないように選ばれたランダムな単一量子ビットパウリで挟みます。多数のランダムなインスタンスにわたって平均すると、任意のノイズチャネルが確率的な *パウリ* チャネルになります。すなわち、そのままでは深さに対して2次的に蓄積するコヒーレントエラーが、線形にしか増えないパウリノイズへと整形されます。ハードウェアのエラーのほとんどは2量子ビットゲートから来るため、トワリングは通常ネイティブな2量子ビットゲートに適用されます。また、パウリチャネルノイズを仮定する PEA と PEC の前提条件でもあります。

![パウリトワリングは、1つの回路を等価なトワリング済み回路のランダムなアンサンブルに置き換える（IBM Quantum docs）。](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/pauli_twirling.svg)

- `twirling.enable_gates` — 2量子ビットゲート層をトワリングする。
- `twirling.num_randomizations` — ランダムなトワリング済みインスタンスの数。
- `twirling.shots_per_randomization` — インスタンスごとのショット数。

#### Twirled readout error extinction（TREX） — *緩和*
TREX は *測定* エラーを対象とします。測定の前にランダムに X を挿入し、その後で古典ビットを反転して戻すことで、読み出しをトワリングします。読み出しエラーがある状態でこれを行うと、読み出しエラーの転送行列が対角化されるため、単純なリスケーリングで反転できます。リスケーリング係数は、ゼロ状態に準備したランダム回路をベンチマークすることで学習され、コストはわずかなキャリブレーション回路です。これはデフォルトでオンになっている唯一の手法です（_resilience level 1_）。

![測定トワリング: 測定前の X ゲートとその後の古典ビット反転は、ノイズがなければ素の測定と等価である（IBM Quantum docs）。](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/measurement_twirling.svg)

- `resilience.measure_mitigation` — TREX をオンにする。
- `resilience.measure_noise_learning.num_randomizations`、`...shots_per_randomization` — 測定ノイズ学習を制御する。

#### ゼロノイズ外挿（ZNE） — *緩和*
ZNE は、いくつかの *増幅された* ノイズレベルで回路を実行し、期待値をゼロノイズへ外挿します。Runtime はデジタルなゲートフォールディングによってノイズを増幅します。これはユニタリー $U$ を $U U^\dagger U$（およびより長いフォールディング）に置き換えて、そのノイズを既知の係数だけ増やし、その結果を選んだ関数形でフィットします。多くの場合精度は向上しますが、不偏であることは *保証されません*。デフォルトでは3つのノイズ係数をサンプリングし、およそ3倍のオーバーヘッドになります。

![ゼロノイズ外挿: デジタルなゲートフォールディングがノイズを増幅し、結果をゼロノイズ極限へ外挿する（IBM Quantum docs）。](https://quantum.cloud.ibm.com/docs/images/guides/error-mitigation-explanation/zne.avif)

- `resilience.zne_mitigation` — ZNE をオンにする。
- `resilience.zne.noise_factors` — 増幅係数（デフォルトは `(1, 3, 5)`）。
- `resilience.zne.extrapolator` — フィット形（例: 線形、指数）。

#### 確率的エラー増幅（PEA） — *緩和*
PEA は ZNE の *ための* より正確な増幅器です。ゲートフォールディングの代わりに、まず各エンタングリング層のトワリング済みパウリ・リンドブラッドノイズモデルを **学習** し、その後、その学習されたモデルから引いた単一量子ビットノイズを確率的に注入することで増幅します。これは合成的なものではなく、実際のデバイスノイズを忠実にスケーリングしたものです。増幅された結果は引き続き ZNE の外挿に渡されるので、PEA は ZNE の上に重ねてオンにします。実用規模の実験では、しばしば最良の増幅器の選択肢になります。

- `resilience.zne_mitigation = True` を `resilience.zne.amplifier = "pea"` とともに使う。

#### 確率的エラーキャンセル（PEC） — *緩和*
PEC もノイズモデルを学習しますが、それを増幅する代わりに *反転* します。平均が逆ノイズチャネルを実装する「反ノイズ（anti-noise）」回路のアンサンブルをサンプリングし、**不偏な** 推定値を与えます。その代償は $\gamma^2$ としてスケールするサンプリングオーバーヘッドであり、回路の深さと総ノイズとともに急速に増大します。だからこそ PEC は適度なサイズの回路に対してのみ実用的です（このコストについては 1.3 で戻ります）。第3章では、_Samplomatic_ を使って PEC を実装し、PEC の発展形である shaded lightcones（SLC）を紹介します。

- `resilience.pec_mitigation` — PEC をオンにする。
- `resilience.pec.max_overhead` — サンプリングオーバーヘッド $\gamma^2$ の上限。上限を超えそうな場合、Runtime は控えめにします。

__注意__: ZNE/PEA と PEC は *相反する* 戦略です（ノイズを増幅するか、それとも打ち消すか）。**同じ** `EstimatorOptions` 上で有効にすることはできません。

<div class="alert alert-block alert-success">

**演習1 — 各手法を設定する。**

1.1.2 節の手法と属性のパスを使って、5つのそれぞれを個別の `EstimatorOptions` オブジェクトに設定し、それらをキー `"dd"`、`"pt"`、`"trex"`、`"zne"`、`"pec"` を持つ辞書 `options_dict` にまとめてください。各手法は個別にチェックされます。合格するには5つすべてが正しくなければならず、grader は正しくないものを報告します。

- **`dd_options`** — 動的デカップリング: `enable = True`、`sequence_type = "XY4"`
- **`pt_options`** — パウリトワリング: `enable_gates = True`、`strategy = "active"`、`num_randomizations = 32`
- **`trex_options`** — TREX: `measure_mitigation = True`
- **`zne_options`** — PEA 付きの ZNE: `zne_mitigation = True`、`noise_factors = [1, 3, 5]`、`amplifier = "pea"`
- **`pec_options`** — PEC: `pec_mitigation = True`、`max_overhead = 100`

</div>

In [ ]:
dd_options = EstimatorOptions()
#TODO: ここにコードを書く #

pt_options = EstimatorOptions()
#TODO: ここにコードを書く #

trex_options = EstimatorOptions()
#TODO: ここにコードを書く #

zne_options = EstimatorOptions()
#TODO: ここにコードを書く #

pec_options = EstimatorOptions()
#TODO: ここにコードを書く #

options_dict = {
    "dd": dd_options,
    "pt": pt_options,
    "trex": trex_options,
    "zne": zne_options,
    "pec": pec_options,
}

In [ ]:
# 解答を採点する
grade_lab3_ex1(options_dict)

In [ ]:
# すべてのラボにわたる進捗を確認する
check_progress()

## 1.2  背後にある代数: ドレスされた層と交換関係

1.1 節のオプションは、回路全体に対して __一度だけ__ 適用されます。それらのいくつかは、共通する基礎的な演算を共有しています。それは、ゲートの論理的な作用は変えないまま、それに付随するノイズを変換するように選ばれた単一量子ビットパウリでゲートの両側を挟む、という演算です。ゲートトワリングはこの演算を直接使います。ZNE/PEA と PEC は、学習するパウリノイズモデルを通して間接的にこれを使います。このように挟まれたゲートを **ドレスされた（dressed）** ゲートと呼びます。

この節では、_ゲートをドレスする_ とはどういうことかを説明します。これは、1.1 節のエラー緩和のスイッチがなぜあのように振る舞うのかを理解するのに役立ちます。また、これは Samplomatic フレームワークにおける、よりきめ細かいボックス単位のツールが使う演算でもあります。このフレームワークによって、これまで見てきた _回路全体のスイッチ_ のアプローチを超えられることを、第2章で見ていきます。

**ドレスされた層** とは、両側を単一量子ビットユニタリーで挟まれた2量子ビットユニタリーです。まず一般的な形で導入し、その後、CZ をパウリでドレスする場合に特化します（これは後で扱うすべての2量子ビット層の設定です）。この節を貫く筋は、パウリと素のゲートとの間の *交換関係*（クリフォード共役）です。それがドレッシング表に構造を与えるものであり、後で Samplomatic がボックスごとに仮想パウリを回路全体に伝播させるために使うのと同じ演算です。

### 1.2.1  ユニタリーをドレスする

任意の $n$ 量子ビットユニタリー $U$ に対して、*ドレスされた* バージョンは

$$\tilde{U} \;=\; V_\text{out}\, U\, V_\text{in},$$

です。ここで $V_\text{in}$ と $V_\text{out}$ は単一量子ビットユニタリーの積（量子ビットごとに1つ）です。ドレッシングがグローバル位相を除いて $V_\text{out}\,U\,V_\text{in} = U$ を満たすとき、それを *不変（invariant）* と呼びます。$U$ の論理的な作用は変わりませんが、$U$ に付随するノイズチャネルは $V_\text{in}$ と $V_\text{out}$ によって *トワリング* されます（[Wallman & Emerson, *Noise tailoring for scalable quantum computation via randomized compiling*, PRA 94, 2016](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325)）。ここから2つの問いが生じます。どのペア $(V_\text{in}, V_\text{out})$ が $U$ を不変にするのか、そしてそれらを体系的にどう列挙するのか、です。

このラボを通して扱うのは、$V_\text{in}$ と $V_\text{out}$ がパウリで、$U$ がクリフォードである場合です。これは IBM ハードウェア上の2量子ビットゲート（CZ、CX、ECR）の設定です。`qiskit.quantum_info` の2つのヘルパーがこれを具体的にします。`Pauli("IX").evolve(U, frame='s')` は正しい符号付きで $U\,P\,U^\dagger$ を返し、`Pauli.equiv(other)` は2つのパウリを全体位相を除いて比較します — これがまさに、私たちが必要とする「グローバル位相を除いて不変」という概念です。

In [ ]:
def dress(U: QuantumCircuit, v_in: str, v_out: str) -> QuantumCircuit:
    """回路  V_out . U . V_in  を QuantumCircuit として構築する。
    """
    n = U.num_qubits
    assert not v_in.startswith(("+", "-", "i")), (
        f"v_in must be an unsigned Pauli string, got {v_in!r}"
    )
    assert not v_out.startswith(("+", "-", "i")), (
        f"v_out must be an unsigned Pauli string, got {v_out!r}"
    )
    assert len(v_in) == len(v_out) == n, "Pauli string length must match U's num_qubits"
    qc = QuantumCircuit(n)
    # Qiskit の文字列 'q_{n-1}...q_0' — 量子ビット順に反復するため逆順にする。
    for q, p in enumerate(reversed(v_in)):
        if p != "I":
            getattr(qc, p.lower())(q)
    qc.compose(U, inplace=True)
    for q, p in enumerate(reversed(v_out)):
        if p != "I":
            getattr(qc, p.lower())(q)
    return qc


def is_invariant(U: QuantumCircuit, v_in: str, v_out: str) -> bool:
    """V_out . U . V_in  がグローバル位相を除いて  U  に等しいとき、かつそのときに限り True。

    *符号なし* のパウリ文字列 v_in, v_out を受け取る。不変性の条件は
    U V_in U† = ±V_out であり、± はドレスされた演算子上のグローバル位相になるので、
    不変性には影響しない。位相を無視して比較する `Pauli.equiv` を使う。
    """
    return Pauli(v_in).evolve(U, frame='s').equiv(Pauli(v_out))

### 1.2.2  CZ の例

$U = \mathrm{CZ}$ に特化します。なぜなら、それがこのラボ全体で使う2量子ビットゲートだからです — 2.6.1 節のイジング回路は完全に CZ ゲートから構成されています。特定のパウリドレッシングが CZ を不変にすることを直接計算で確認した後、16 個すべての2量子ビットパウリを CZ を通して伝播させ、完全な写像 $V_\text{in} \mapsto V_\text{out}$ を構築します。各 $V_\text{in}$ に対して、有効な $V_\text{out}$ は（符号を除いて）ちょうど1つ存在します。

下に出力する表には、2つの異なる情報が含まれており、これらを混同してはいけません。*ドレッシング写像* は、$V_\text{out}$ として物理的に適用すべき **符号なし** のパウリを記録するもので、`dress()` が消費するものです。

**代数的な符号** は、$\mathrm{CZ}\cdot V_\text{in}\cdot \mathrm{CZ}$ が $+V_\text{out}$ に等しいか $-V_\text{out}$ に等しいかを定めるもので、ゲートの不変性には **影響しません**（符号はグローバル位相として吸収されます）が、1.2.3 節の伝播規則には **影響します**。

表そのものにはきれいな構造があることが分かります。$Z$ 因子はそのまま CZ を通過し、片方の量子ビット上の $X$ または $Y$ 因子は **もう一方の** 量子ビット上に $Z$ を拾い、ペア $XY \leftrightarrow YX$ はマイナス符号を拾います。これは下の出力の一番右（符号）の列で見ることができます。

In [ ]:
# 素の CZ 層
cz_layer = QuantumCircuit(2, name="CZ")
cz_layer.cz(0, 1)
cz_layer.draw("mpl")

In [ ]:
# CZ の前に量子ビット0に X を適用し、その後に量子ビット0の X + 量子ビット1の Z を適用する。

print("V_in='IX', V_out='ZX' leaves CZ invariant?",
      is_invariant(cz_layer, "IX", "ZX"))

In [ ]:
# パウリ代数から2つの関連するオブジェクトを構築する:
#   cz_twirl_map       : V_in -> V_out  （符号なし — ドレッシングとして適用するもの）
#   cz_commutation_map : V_in -> (sign, V_out)  （符号付き — 伝播規則のため）

PAULIS = ["I", "X", "Y", "Z"]


def split_sign(label: str) -> tuple[int, str]:
    """符号付きパウリラベルを (+/-1, unsigned_label) に分割する。"""
    if label.startswith("-"):
        return -1, label[1:]
    if label.startswith("+"):
        return +1, label[1:]
    return +1, label


cz_twirl_map       = {}
cz_commutation_map = {}

for p1 in PAULIS:         # 量子ビット1上のパウリ（文字列の左端）
    for p0 in PAULIS:     # 量子ビット0上のパウリ（文字列の右端）
        v_in   = p1 + p0
        signed = Pauli(v_in).evolve(cz_layer, frame="s").to_label()
        sign, bare = split_sign(signed)
        cz_twirl_map[v_in]       = bare
        cz_commutation_map[v_in] = (sign, bare)

print(f"Total Pauli dressings for CZ: {len(cz_twirl_map)}  (expected 16)")
print()
print(f"  {'V_in':<6}{'V_out (apply)':<16}{'algebraic sign'}")
print(f"  {'-'*6}{'-'*16}{'-'*14}")
for v_in, v_out in cz_twirl_map.items():
    sign, _ = cz_commutation_map[v_in]
    sign_str = "+" if sign > 0 else "−"
    print(f"  {v_in:<6}{v_out:<16}{sign_str}")

### 1.2.3  交換関係: パウリはどのようにクリフォードを伝播するか

1.2.2 節の表の構造は偶然ではありません。任意のパウリ $P$ とクリフォード $U$ に対して、

$$U\,P\,U^\dagger \;=\; \pm\,P',$$

です。ここで $P'$ は別のパウリです。これはクリフォード群を定義する性質です。$V_\text{in}$ が与えられたときに $V_\text{out}$ を求めることは、$V_\text{in}$ を $U$ を通して伝播させ、符号を含めてその結果を読み取ることに等しいです。これは、Samplomatic が第2章で仮想パウリをゲートボックスを横切って伝播させるたびに使う演算です。`Pauli.evolve(U, frame='s')` はそれを実装する Qiskit のヘルパーであり、`.to_label()` は `"-YX"` のような符号付きパウリ文字列を返します。

同じヘルパーを CX に対して実行すると、CX の制御/標的の役割に由来する非対称性を持つ、構造的に似た表が得られます。

In [ ]:
# 同じ Pauli.evolve(..., frame="s") ヘルパーを、2つの異なるゲートに適用する。
# 伝播規則はゲートによって固定されるので、CZ と CX は異なる表を与える。
cx_layer = QuantumCircuit(2, name="CX")
cx_layer.cx(0, 1)  # 制御 = q0、標的 = q1

reps = ["IX", "XI", "IZ", "ZI", "IY", "YI"]

print("CZ . P . CZ  (symmetric in the two qubits):")
for p in reps:
    print(f"  CZ . {p} . CZ  =  {Pauli(p).evolve(cz_layer, frame='s').to_label()}")

print("\nCX . P . CX  (control = q0, target = q1):")
for p in reps:
    print(f"  CX . {p} . CX  =  {Pauli(p).evolve(cx_layer, frame='s').to_label()}")

上の2つの表が異なるのは、伝播規則がゲートによって固定されているからです。CZ は対称的です。どちらかの量子ビット上の $Z$ はそのまま通過し、$X$ または $Y$ はもう一方の量子ビット上に $Z$ を拾います。CX はそうではありません。標的量子ビット上の $X$ と制御量子ビット上の $Z$ はそれぞれ両方の量子ビットに広がりますが、制御量子ビット上の $X$ と標的量子ビット上の $Z$ は変わらずに通過します。

手順はどちらの場合も同じです。パウリをゲートで共役させ、符号付きの結果を読み取ります。これは、このラボのトワリングに基づくすべての手法が拠って立つ演算です。1.1.2 節の `twirling.enable_gates` スイッチから、第2章で紹介する Samplomatic のボックスごとの `Twirl` アノテーションまで、すべてがそうです。

これらの手法を分けるのは、その演算が適用されるスケールです。`EstimatorOptions` のスイッチは、それを回路全体に一度だけ適用します — これは最も粗い選択肢であり、個々の層やオブザーバブルを異なる方法で扱う必要があるようないくつかの緩和タスクではうまくいきません。1.3 節では、Runtime オプションができることとできないこと、そしてそのリソースコストを整理します。これが、このラボの残りの部分で使う、よりきめ細かいツールへの動機付けになります。

## 1.3  なぜ構造を意識した緩和が必要なのか

前の節では、`EstimatorOptions` のスイッチができることと、その背後にある代数を示しました。この節では、回路全体のスイッチが不十分になる場面、それが表現できないエラー緩和タスク、そして回路が大きくなるにつれて発生するリソースコストを見ます。これを Samplomatic を使って行います。これは、このラボの残りの部分の土台となるボックス単位のツールです。

#### Runtime オプションができないこと

Runtime オプションは、「すべてのアイドルウィンドウに DD を挿入する」や「すべての2量子ビットゲートをトワリングする」のような **回路全体のスイッチ** を公開しています。それぞれは単一の回路全体のポリシーによって適用され、多くの小さなワークロードには十分です。PEA と PEC は内部的には *エンタングリング層ごとに* ノイズモデルを学習しますが、API は層ごとの制御を公開していません。層の学習されたノイズを検査したり、上書きしたり、異なる層に異なる戦略を適用したりすることはできません。別の機能はまったく手の届かないところにあります。それは、回路を緩和する代わりに、オブザーバブルそのものを書き換えて学習されたノイズを吸収することです（第3章で __propagated noise absorption__（PNA）とともに見ます）。

第2章では、このよりきめ細かい制御を提供するツールを紹介します。それは *ボックス* — `Twirl`、`InjectNoise`、`ChangeBasis` などのディレクティブでアノテーションされた回路の一部分 — と、`Sampler` および `Estimator` のボックスを意識した対応物である `NoiseLearnerV3` と `Executor` です。

#### 隠れたコスト: サンプリングオーバーヘッド

これらのスイッチをオンにすることはただではなく、その代金は量子ビットではなく **ショット** で支払われます。エラー緩和は _空間_ のオーバーヘッドを _時間_ のオーバーヘッドと引き換えにします。（誤り訂正のように）追加の物理量子ビットを使う代わりに、多数のノイズを含む回路を実行して事後処理します。このコストの一部は、ZNE の `noise_factors`、`twirling.num_randomizations`、TREX の読み出しキャリブレーションバッチのように、オプションから読み取れる単純な乗数です。たとえば、32 のランダム化で3つの ZNE 係数を使うと、緩和なしの単一実行に対してすでに `3 × 32` 程度のバリアントを実行することになります。

オプションから *読み取れない* もので、実用規模の回路を制限するものがあります。それは、確率的手法の **サンプリングオーバーヘッド** が *指数的* に増大することです。PEC の場合、ランタイム $J$ は次で与えられます。
$$J = \bar{\gamma}^{\,nd}\,\beta^{d},$$
ここで $n$ は量子ビット数、$d$ は深さです。底 $\bar{\gamma}$ は *学習された* ノイズモデルによって決まります。これは品質の指標と考えることができます。ゲートエラーが低いほど $\bar{\gamma}$ は小さくなります。$\beta$ は回路の層ごとの時間（速度の指標）です。$\bar{\gamma}$ が指数的に効いてくるため、たとえそれをわずかに減らすだけでも（すなわちゲートエラーを低くするだけでも）、$\bar{\gamma}^{nd}$ の大きさは劇的に減少します。

<img src="https://research-website-prod-cms-uploads.s3.us.cloud-object-storage.appdomain.cloud/New_Fig_Gamma_Epsilon2_Tex_100q_trotter_a712254dca.jpg" width="450" alt="100 量子ビットのイジングトロッター回路に対する PEC 回路オーバーヘッドの見積もり">

*図1: イジングスピン鎖の時間発展を表す、深さ 400 と 4000 の 100 量子ビットトロッター化回路に対する PEC 回路オーバーヘッドの見積もり。赤い点線は、固定 1kHz のサンプリングレートを仮定した場合の1日のランタイムを示す。K. Temme, E. van den Berg, A. Kandala, and J. Gambetta, "[Error mitigation is the path to quantum computing usefulness](https://www.ibm.com/quantum/blog/gammabar-for-quantum-advantage)," IBM Quantum blog, 2022 より転載。© IBM。*

上の図は、このラボが使うのと同じ種類の回路 — イジングハミルトニアンのトロッター発展（2.6 節で紹介します）に対するものです。図は、2量子ビットゲートエラーが下がるにつれてオーバーヘッドが桁違いに下がることを示しています。`EstimatorOptions` で使える回路全体のスイッチには、$\bar{\gamma}$ を決める層ごとのノイズを見たり、それに作用したりする方法がありません。これがこのラボの残りの部分が埋めるギャップです。第2章では `NoiseLearnerV3` で層ごとのノイズを *測定* し、第3章では層とオブザーバブルを個別に扱うことでオーバーヘッドを *削減* します（PNA と SLC）。

## 第2章 — Samplomatic と NoiseLearner

個々の層やゲートのレベルでノイズを緩和するには、3つのものが必要です。回路の各部分を別々に指定する方法、各部分のノイズの測定、そして緩和されたプログラムを実行して結果を集める方法です。Samplomatic、`NoiseLearnerV3`、`Executor` プリミティブがこの3つを供給します。これらは合わさって Qiskit Runtime の [directed execution model](https://quantum.cloud.ibm.com/docs/en/guides/directed-execution-model) を形成します。これは緩和の意図をクライアント側で捉え、回路バリアントの生成をサーバーへ移すものです。

[Samplomatic](https://github.com/Qiskit/samplomatic) は、**ボックス** と **アノテーション** を通して回路の各部分を指定します。ボックスは、1つのゲート、1つの層、または最終測定など、回路の1領域を1つのまとまりとして区切ったものです。アノテーションは、ボックスに付けられた指示です。`Twirl` はボックスのドレッシングをランダム化し、`InjectNoise` は学習されたノイズチャネルがここに適用されることを宣言し、`ChangeBasis` は測定を回転させます。指示は回路全体ではなくボックスに付けられるので、各ボックスを別々に扱えます。次に `build` ステップが、アノテーションされた回路をパラメトリックな *テンプレート* と *samplex* に変換します。samplex は、実行時にボックスのパラメーターを埋めるためのレシピです。

[`NoiseLearnerV3`](https://quantum.cloud.ibm.com/docs/en/guides/noise-learning) は測定を供給します。バックエンド上で特性評価回路を実行し、Samplomatic が使うのと同じボックスで指定される各ボックスに対して [パウリ・リンドブラッドノイズモデル](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/utils-noise-learner-result-pauli-lindblad-error) を返します。この2つは1つのパターンでかみ合います。ボックスは `InjectNoise` によってノイズモデルを期待することを *宣言* し、`NoiseLearnerV3` が測定した具体的なモデルが、`InjectNoise` の参照によって照合されて実行時に差し込まれます。

[`Executor`](https://quantum.cloud.ibm.com/docs/en/api/qiskit-ibm-runtime/executor) プリミティブは、緩和されたプログラムそのものを実行します。これと `NoiseLearnerV3` はどちらもバックエンドにジョブを投入しますが、目的が異なります。`NoiseLearnerV3` は *ノイズを測定する* ために回路を実行し、`Executor` はテンプレートと samplex から構築されたボックスを意識したプログラムを実行して *計算の結果を生成* します。これは `Sampler` と `Estimator` の [ボックスを意識した対応物](https://quantum.cloud.ibm.com/docs/en/guides/qiskit-runtime-primitives) です。__ボックス化（Boxing）__ は、この3つのステップすべてに共通する構造です（ボックスごとの緩和をどう指定するか、ボックスごとのノイズをどう報告するか、そして Executor が何を実行するか）。そして、第3章のアドオンである propagated noise absorption（PNA）と shaded lightcones（SLC）が、1つの層または1つのオブザーバブルを一度に緩和するために作用する対象でもあります。

以下の節では、小さな2量子ビットのトイモデルに対して、各ツールを一歩ずつ紹介します。その後、2.6 節でそれらを1次元イジング鎖の上でまとめます。これは第3章が続けて扱う系です。

## 2.1  ボックスと `Twirl` アノテーション

まず回路をボックス化することから始めます。とてもシンプルな例から始めましょう。単一の CZ ゲートを持つ2量子ビット回路です。`.box` メソッドを使って CZ ゲートの周りにボックスを作り、それが `Twirl` オブジェクトでアノテーションされていることを伝えます。

1.1.2 節では、エラー緩和の重要なツールとしてパウリトワリングを紹介しました。量子回路をパウリトワリングすると、回路のノイズが _確率的なパウリチャネル_ に整形されます。これは、1つの回路をランダムな回路のアンサンブル、すなわち _ランダム化（randomization）_ に置き換えることで行われます。重要なのは、トワリングだけでは回路のエラーを緩和することは期待されておらず、むしろ他の技術によって効果的に緩和できるようにノイズを整形する、ということです。

`Twirl` アノテーションは、ボックスを _トワリング_ すべきであることを示すために使われます。

In [ ]:
toy = QuantumCircuit(2)
with toy.box(annotations=[Twirl()]):
    toy.cz(0, 1)

toy.draw("mpl")

回路図は、CZ ゲートを包むボックス（赤い四角）を示していますが、`Twirl` アノテーションそのものは図には現れません。これはボックスに付けられたメタデータです。回路の命令を反復処理することで読み返すことができます。

In [ ]:
for idx, instruction in enumerate(toy):
    print(f"Box {idx}: annotations = {instruction.operation.annotations}")

上に出力された命令は、次のことを教えてくれます。

この回路には1つのボックスがあり、単一のアノテーション `Twirl(group='pauli', dressing='left', decomposition='rzsx')` を持っています。これらはデフォルトの `Twirl` 設定です。分解してみましょう。
- `group='pauli'`: パウリ群のトワリングを適用する
- `dressing='left'`: ランダムなドレッシングはボックスの左側にある
- `decomposition='rzsx'`: ドレッシングがハードウェアのネイティブな `rz` と `sx` ゲートにコンパイルされることを記録する。

ボックスは回路の *どの* 部分をトワリングするかを示し、アノテーションは *どのように* するかを記録します。この意図を実行可能な回路に変えるのが次のステップです。

### 2.1.1  テンプレート

Samplomatic のワークフローの次のステップは、テンプレート回路と samplex を *ビルド* することです。`build(boxed_circuit)` メソッドは、アノテーションされた回路を2つのオブジェクトに変換します。*テンプレート*（任意の単一のランダム化を実現するのに十分な自由なゲートを持つパラメトリックな `QuantumCircuit`）と *samplex*（実行時にそれらのパラメーターをどう埋めるかを記述するレシピ）です。

`build` を使おうとするときによくあるエラーは、ドレッシングの行き先がない場合です。これを下のセルで示します。

In [ ]:
try:
    template_circuit, samplex = build(toy)
except Exception as e:
    print(f"{type(e).__name__}: {e}")

`build` が失敗するのが分かります。`Twirl` アノテーションは、ランダムパウリのドレッシングをボックスの _左_（入力）側に置くよう Samplomatic に指示します。しかし、ボックスの論理的な作用が変わらないようにするには、そのパウリは CZ を通って伝播し、後続のどこかのボックスが吸収しなければならない *仮想ゲート* として _右_（出力）側から出てこなければなりません。ここでは回路がボックスの直後で終わってしまうので、それらの右向きの仮想ゲートには _着地する場所がなく_、その結果 `SamplexBuildError` が得られます。エラーは量子ビット `[0, 1]` 上の終端されていない仮想ゲートを報告します。`Twirl` ボックスは常に *コレクター*、すなわちそれらを受け取る別のボックスを必要とします。次のセルでは、測定ボックスとしてそれを1つ追加します。そうすれば `build` を成功させることができます。

In [ ]:
toy = QuantumCircuit(2)
with toy.box(annotations=[Twirl()]):
    toy.cz(0, 1)
with toy.box(annotations=[Twirl()]):
    toy.measure_all()

template_circuit, samplex = build(toy)
template_circuit.draw("mpl", fold=-1)

測定をそれ自身の `Twirl` ボックスで包むと、右側のドレッシングに着地する場所ができます。CZ ボックスから出てくる仮想ゲートが測定ボックスに吸収され、同時に読み出しもトワリングされます。`build` が成功し、`template` とその `samplex` を返します。

テンプレートの回路図では、単一量子ビットゲートは `rz` と `sx` に分解され、自由なパラメーターとして残されています。`samplex` は、ランダム化が引かれるたびにそれらのパラメーターを具体的なランダムパウリの割り当てで埋めるレシピを保持しています。1つのテンプレートと1つの samplex が、ランダム化された回路のアンサンブル全体を代表します。

### 2.1.2  samplex


上で見たように、`build` は2つのオブジェクトを返します。_テンプレート_ と _samplex_ です。samplex は実行時のレシピです。各ランダム化について、テンプレート回路に与えるパラメーター値と、測定されたビット列に適用する事後処理を決める、入力と出力の [有向非巡回グラフ](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.dagcircuit.DAGCircuit)（DAG）です。

In [ ]:
print(samplex)
fig = samplex.draw()
fig.show(renderer="notebook")
print(type(fig))

上の出力テキストは samplex の入力と出力を示し、DAG 図はそれらをグラフとして示しています。**Inputs** のリストは空です。これは、`Twirl` アノテーションのみを持つ samplex は実行時データを必要としないためです。`InjectNoise` アノテーションが追加された場合（2.4 節で見ます）とは異なります。2つの **Outputs**、`parameter_values` と `measurement_flips.meas` は、図の中のコレクターノード（蝶ネクタイ形）です。

DAG 図は量子回路では **ありません**。これは samplex の古典的なデータフローグラフです。テンプレート（2.1.1 節で上に描いたもの）が回路であり、samplex は各ランダム化についてそのパラメーター枠に何を埋めるかを計算します。したがって、samplex 図のノードは計算ステップであり、エッジはステップ間で古典データのレジスターを運びます。

samplex には3種類のノードが現れます。
- **赤い星** は *サンプリング* ノードです。`Twirl` ボックスごとに1つあり、そのボックスのランダムパウリを引きます。私たちのトイ例には2つのボックス、CZ ボックスと測定ボックスがあります。
- **緑の円** は処理ステップです。パウリを CZ の先へ伝播させたり（1.2 のクリフォード共役を、いまはグラフのノードとして可視化したもの）、レジスターをスライスしたり結合したりします。
- **蝶ネクタイ形** は、データを出力に集める *コレクター* です。**青** のものは `parameter_values`（`[num_randomizations, 12]`、テンプレートの自由なパラメーター）へ、**紫** のものは `measurement_flips.meas`（`[num_randomizations, 1, 2]`、測定トワリングを打ち消すビット反転）へ集めます。

次はサンプリングのステップです。これを行うために、`samplex.sample(...)` メソッドを使い、ランダム化の数 `num_randomizations` を指定します。`samplex.sample(...)` を呼ぶたびに、トワリング分布から `num_randomizations` 個の独立したランダムパウリの集合が引かれます。それらは、回路が必要とする形にすでに変換されて返されます。テンプレートを埋める `parameter_values` と、後で読み出しを補正する `measurement_flips` です。

In [ ]:
outputs = samplex.sample({}, num_randomizations=5)

print("parameter_values.shape     :", outputs["parameter_values"].shape)
print("measurement_flips.meas.shape:", outputs["measurement_flips.meas"].shape)
print()
print("First two parameter draws:")
print(outputs["parameter_values"][:2])
print()
print("Bit-flip corrections:")
print(outputs["measurement_flips.meas"][:, 0, :])

ここでは5つのランダム化をサンプリングしました。出力は2つのキー `parameter_values` と `measurement_flips` を持つ辞書です。トイモデルのテンプレートには12個のパラメーターがあり、測定トワリングには2ビットあるので、出力の形状は期待どおりです。テンプレート用に12個のパラメーターの集合が5組、測定トワリング用に2個のビット反転補正の集合が5組あります。

これは純粋に古典的なステップです。まだ回路は実行されていません。テンプレートは固定されたままで、ランダム化ごとに変わるのはパラメーター値と測定反転だけです。埋め込まれた回路をハードウェアで実行するのは `Executor` の仕事で、2.5 節で扱います。

ここから自然に2つの問いが生じます。ランダム化をいくつ引くか、そして `with circuit.box(...)` ブロックを手で書かずに回路をボックス化する方法です。次の節で両方に答えます。

## 2.2  ランダム化とボックス化パスマネージャー

1つ目の問いは、ランダム化をいくつ引くかです。これを制御する2つのパラメーターがあり、混同しやすいものです。`num_randomizations`（トワリング分布から引かれる独立したランダム回路の数）と `shots_per_randomization`（各回路の中で取るショット数）です。これらは互いにトレードオフの関係にあります。ある比を超えると、トワリングによる分散がショットごとの統計ノイズを支配するので、ショットを増やすよりランダム化を増やすほうが効果的になります。

2つ目の問いは、`with circuit.box(...)` ブロックを手で書かずに回路をボックス化する方法です。これは [_ボックス化パスマネージャー_](https://qiskit.github.io/samplomatic/guides/transpiler.html) で行えます。`generate_boxing_pass_manager(...)` は、回路を自動的にボックス化してアノテーションするトランスパイラーのパスです。この節では、そのオプションのうち3つ — `enable_gates`、`enable_measures`、`twirling_strategy` を使います。`inject_noise_*` オプションは 2.3 節で、測定アノテーションは 2.4 節で扱います。

2.1 節と同じトイ回路 — CZ に続いて測定 — を、今度は手書きのボックスブロックの代わりにパスマネージャーで自動的にボックス化してみましょう。

In [ ]:
# 素のトイ回路を作る
raw_toy = QuantumCircuit(2, 2)
raw_toy.cz(0, 1)
raw_toy.measure([0, 1], [0, 1])

# ラボの最初で定義した isa_pm を使ってトイ回路をトランスパイルする
raw_toy_isa = isa_pm.run(raw_toy)

# ボックス化パスマネージャーを作る
twirl_only_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    twirling_strategy="active",
)

# ボックス化パスマネージャーを使ってトイ回路をボックス化する
boxed_toy = twirl_only_pm.run(raw_toy_isa)
boxed_toy.draw("mpl", idle_wires=False)

ボックス化パスマネージャーは、素の回路（ボックスなし）を受け取り、2.1 節で手で作ったのと同じボックス化された回路を自動的に生成します。

- `enable_gates=True` は2量子ビットゲートを `Twirl` ボックスで包む
- `enable_measures=True` は測定をボックスで包む
- `twirling_strategy="active"` は各ボックスが作用する量子ビットのみをトワリングする

回路図は、結果として得られる2つのボックスを赤で示しています。それぞれがデフォルトの `Twirl` アノテーション（回路図には現れません）を持っています。

多くの層を持つ回路では、ボックス化パスマネージャーを使うのが実用的なボックス化の方法です。すべての層に対して `with circuit.box(...)` を手で書くのは、うまくスケールしません。

## 2.3  `InjectNoise` アノテーションと `NoiseLearnerV3`

2.2 節では、__ボックス化パスマネージャー__ を使って回路を自動的にボックス化できることを示しました。ここまでは `Twirl` アノテーションでアノテーションされたボックスしか見ていません。パウリトワリングは、各層のコヒーレントエラーを _確率的なパウリチャネル_ に変えます。これは、ノイズを扱いやすくするので強力です。疎なパウリチャネルは、任意のノイズプロセスよりもはるかに単純な対象です。しかし、トワリングだけではノイズは除去されません。

この節では、`InjectNoise` アノテーションを紹介します。このアノテーションを `NoiseLearnerV3` とともに使って、各ボックスのノイズを学習します。これによって、第3章で __PNA__ や __SLC__ のような高度なエラー緩和の手法を実装できるようになります。

この節は3つのステップで進みます。まず 2.3.1 節で、すべてのゲートボックスが `InjectNoise` アノテーション — `ref` 文字列でキー付けされた宣言済みの枠 — を持つように、ボックス化パスマネージャーを拡張します。次に 2.3.2 節で、`NoiseLearnerV3` をバックエンド上で実行し、各枠を学習されたパウリ・リンドブラッドモデルで埋めます。最後に 2.3.3 節で、返されたノイズモデルを検査し、どのエラー生成子が層を支配しているかを見ます。

### 2.3.1  枠を宣言する: 各ゲートボックスの `InjectNoise`

下のセルでは、まず `noise_learning_boxing_pm` と名付ける新しいボックス化パスマネージャーを生成します。上のボックス化マネージャー `twirl_only_pm` で使ったオプションを維持し、2つの新しい `inject_noise_*` オプションを追加して、すべてのゲートボックスが `Twirl` に加えて `InjectNoise` アノテーションを持つようにします。それから、このボックス化パスマネージャーをトイ回路に適用します。

次に `find_unique_box_instructions` が、回路から *相異なる* ゲート層を抽出します。これらは `NoiseLearnerV3` が実際に特性評価する層です。というのも、（単一量子ビットのドレッシングを除いて）等価なボックスは同じノイズモデルを共有するからです。これは、等価なボックスが多い回路では特に効率的です。ノイズ学習は _ユニークな_ 層のノイズだけを学習すればよいからです。

In [ ]:
noise_learning_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="no_modification",
)

boxed_toy = noise_learning_boxing_pm.run(raw_toy_isa)

unique_toy_layers = find_unique_box_instructions(
    boxed_toy,
    normalize_annotations=None,
    undress_boxes=True,
)


前と同じように、このボックス化パスマネージャーは2つのボックスを生成します。CZ 層用の1つと測定用の1つです。今回、CZ ゲートボックスは（`inject_noise_targets="gates"` による）`InjectNoise` と `Twirl` アノテーションを持ち、測定ボックスは `Twirl` だけを保ちます。

`inject_noise_strategy="no_modification"` というオプションは、（`find_unique_box_instructions` による）すべての等価なボックスに _同じ_ ref を持つ inject noise アノテーションが割り当てられることを意味します。`ref` は、ノイズを注入するもとになるパウリ・リンドブラッド写像を一意に識別する `string`（文字列）です。これは、すべての同一のボックスに同じラベルを割り当てるものと考えることができます。

第3章では、このオプションを `inject_noise_strategy=uniform_modification`（PNA）と `inject_noise_strategy=individual_modification`（SLC）に変える必要があります。

ボックス化パスマネージャーの詳細については、[こちら](https://qiskit.github.io/samplomatic/api/auto/samplomatic.transpiler.generate_boxing_pass_manager.html#samplomatic.transpiler.generate_boxing_pass_manager) のドキュメントを参照してください。

In [ ]:
for i, inst in enumerate(unique_toy_layers):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a else "(none)"
    print(f"Layer {i}: ref = {ref}")
    display(inst.operation.body.draw("mpl", fold=-1, idle_wires=False))

上のセルでは、各ユニークな層について `ref` 文字列を出力し、対応する層の回路図を描いています。

`InjectNoise` アノテーションを持つ各ユニークな層は、それ自身の `ref` 文字列を持ちます。上で見られるように、CZ ボックスは `ref` を持ちますが、`InjectNoise` アノテーションを持たない測定層は持ちません。

`ref` は、`NoiseLearnerV3` が学習後に各ユニークなボックスのノイズを報告するために使うラベルです。同じ `ref` ラベルが、実行時に `InjectNoise` が学習されたノイズモデルを再び見つけるためにも使われます。

これで枠は宣言されましたが、まだ空です。次の小節では、`NoiseLearnerV3` がそこに *何を* 書き込むのかを説明します。

#### `NoiseLearnerV3` が学習するもの

`NoiseLearnerV3` は、各層を疎な *パウリ・リンドブラッド* チャネルとして特性評価します。

$$\Lambda \;=\; \exp(\mathcal{L}), \qquad \mathcal{L}(\rho) \;=\; \sum_k \lambda_k\,(P_k\,\rho\,P_k - \rho),$$

これは van den Berg, Minev, Kandala & Temme が [*Probabilistic error cancellation with sparse Pauli–Lindblad models on noisy quantum processors*, Nat. Phys. **19**, 1116–1121 (2023)](https://www.nature.com/articles/s41567-023-02042-2) で定式化したモデルに従っています。ここで $\mathcal{L}$ はスーパー演算子（密度行列 $\rho$ に作用する）であり、$\Lambda = e^{\mathcal{L}}$ はその形式的な指数です。生成子 $\{P_k\}$ はエルミートなパウリ（$P_k^\dagger = P_k$、$P_k^2 = I$）なので、散逸子は右側の形に簡約されます。

生成子の集合は、層の接続性によって固定されます。関与する各量子ビットは3つの1体パウリ $X_i, Y_i, Z_i$ を寄与し、各ゲートエッジは9つの非恒等2体パウリ $\{XX, XY, \ldots, ZZ\}$ を寄与します。たとえば、量子ビット $(0,1)$ 上の単一の CZ は、そのモデルに $2\times 3 + 9 = 15$ 個の生成子を持ちます。これらの生成子のレートが層のノイズの構造を定めるものであり、`NoiseLearnerV3` はこれらの生成子レートの値を学習します。2.3.2 節では、トイ回路についてレートを学習し、2.3.3 節ではそれらを検査して、ノイズの構造への洞察を得ます。

_注意_: このモデルは次のものを捉えません。
- __コヒーレントエラー__: コヒーレントエラーはすでに確率的なパウリノイズにトワリングされていると仮定しています
- __層をまたぐクロストーク__: 層の接続性の外にある生成子は現れません。たとえば、生成子 $X_0 X_2$ は、量子ビット $q_0$ と $q_2$ に作用するゲートを持たないどの層にも現れません。

ノイズ学習は、精度と QPU 時間を引き換えにする3つのオプションで設定されます。`num_randomizations`（設定ごとの独立したランダムベンチマーク回路の数）、`shots_per_randomization`（回路ごとのショット数）、そして `layer_pair_depths`（層が繰り返される深さ $d$。学習は、得られた忠実度と深さの減衰をフィットして各生成子のレートを抽出します）です。

### 2.3.2  トイ回路でノイズ学習ジョブを実行する

層をボックス化して宣言したので、それらをバックエンドに送ります。この最初のトイ回路の例では、QPU 時間を短く保つために小さな設定（`num_randomizations=5`、`shots_per_randomization=20`、`layer_pair_depths=[1, 2]`）を選びます。

_注意:_
以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `TOY_NOISE_LEARN_JOB_ID` パラメーターに貼り付け、`SUBMIT_TOY_NOISE_JOB = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは3秒です（ibm_pittsburgh でテスト済み）。_ 上の使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
TOY_NOISE_LEARN_JOB_ID = None   # paste job_id here on re-run
SUBMIT_TOY_NOISE_JOB   = False   # set True to submit a fresh learning job

nl_options = NoiseLearnerV3Options(
    num_randomizations=5,
    shots_per_randomization=20,
    layer_pair_depths=[1, 2],
)
learner = NoiseLearnerV3(backend, nl_options)
learner.options.environment.job_tags = ["qgss26"]

if TOY_NOISE_LEARN_JOB_ID is not None:
    learner_job = service.job(TOY_NOISE_LEARN_JOB_ID)
    print(f"Re-using saved job: {TOY_NOISE_LEARN_JOB_ID}")
elif SUBMIT_TOY_NOISE_JOB:
    learner_job = learner.run(unique_toy_layers)
    TOY_NOISE_LEARN_JOB_ID = learner_job.job_id()
    print(f"Submitted: {TOY_NOISE_LEARN_JOB_ID}")
else:
    print("Set SUBMIT_TOY_NOISE_JOB=True to submit a fresh job, "
          "or paste a saved job id into TOY_NOISE_LEARN_JOB_ID and re-run.")



ノイズ学習ジョブを投入した後、下のセルを実行してジョブの状態を確認してください。`QUEUED`、`RUNNING`、`DONE` のいずれかになるはずです。ジョブが完了したら、次のセルに進んでください。

In [ ]:
learner_job = service.job(TOY_NOISE_LEARN_JOB_ID)
print(f"{TOY_NOISE_LEARN_JOB_ID}  (status: {learner_job.status()})")

In [ ]:
if learner_job.status() == "DONE":
    toy_noise_result = learner_job.result()
else:
    print(f"Not done yet (status={learner_job.status()}). Re-run cell when DONE.")

In [ ]:
if 'toy_noise_result' in dir() and toy_noise_result is not None:
    toy_refs_to_noise = toy_noise_result.to_dict(unique_toy_layers, require_refs=False)
    print(f"toy_refs_to_noise has {len(toy_refs_to_noise)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


上のセルでは、`to_dict` メソッドを使って、ノイズ学習ジョブの結果を `toy_refs_to_noise` に集めます。学習された層ごとに1つのパウリ・リンドブラッドモデルがあり、それぞれがボックス化された回路で使われたのと同じ `ref` でラベル付けされているはずです。トイ回路には `InjectNoise` アノテーションを持つ層（CZ 層）が1つしかないことが分かっています。したがって、ノイズ学習ジョブは単一の学習されたノイズモデルを返すと期待されます。実際にそうなっていることが分かります。

これで枠は埋まりました。実際に何が学習されたかを見るために、2.3.3 節では、単一の CZ 層について学習されたノイズモデルを検査します。

### 2.3.3  学習されたノイズモデルを検査する

下のセルでは、`PauliLindbladMap.to_sparse_list()` メソッドを使って、ノイズモデルを `(pauli, qubits, rate)` タプルの列として返します。$\mathcal{L}$ の非ゼロの生成子ごとに1つのタプルがあります。

これらのタプルをレートでソートして、どのエラーチャネルが CZ 層を支配しているかを示し、上位10個の生成子を出力します。この情報は、層ごとのすべてのエラー緩和戦略が、予算をどこに使うかを決めるために用いる入力です。これが PNA、PEC、SLC についてどう実装されるかは第3章で見ます。

In [ ]:
if "toy_refs_to_noise" not in globals() or not toy_refs_to_noise:
    print("No learned toy noise model yet. Run the noise-learning result cell above first.")
else:
    ref0, plm0 = next(iter(toy_refs_to_noise.items()))
    generators = plm0.to_sparse_list()
    generators.sort(key=lambda g: -abs(g[2]))

    print(f"Layer ref = {ref0}, total generators = {len(generators)}\n")
    print(f"  {'Pauli':<6}{'qubits':<14}{'rate':>10}")
    print(f"  {'-'*6}{'-'*14}{'-'*10}")
    for pauli, qubits, rate in generators[:10]:
        print(f"  {pauli:<6}{str(qubits):<14}{rate:>10.4e}")


もう1つ紹介すべきアノテーションが残っています。非 $Z$ 基底での測定のための `ChangeBasis` です。これは次の 2.4 節で扱います。

## 2.4  `ChangeBasis` アノテーション

`ChangeBasis` は、非 $Z$ 基底での測定に必要です。これは `InjectNoise` と同じパターンに従います。アノテーションは静的に付けられ、実行時に供給されます。これは測定ボックスに付けられ、その値は samplex の `basis_changes` 入力を通して実行時に束縛されます。

第2章の残りの部分では `ChangeBasis` を使いません。2.6 節で考えるオブザーバブルは計算基底で対角だからです。しかし第3章では、PNA が非 $Z$ 項を持つノイズ緩和オブザーバブル $\tilde{O}$ を導入すると、これを使います。

下のセルは、`measure_annotations="all"` が測定ボックスに `Twirl` と `ChangeBasis` の両方を追加することを示しています。これによって `samplex.inputs()` に `basis_changes.<ref>` 枠が追加されます。

In [ ]:
# 測定ボックスにも ChangeBasis を
# アノテーションするボックス化パスマネージャー。
change_basis_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="no_modification",
)

demo_boxed = change_basis_pm.run(raw_toy_isa)
_, demo_samplex = build(demo_boxed)
print(demo_samplex)

samplex は、2.1.2 節でボックスに `Twirl` アノテーションしかなかったときに見たような何もない状態ではなく、2つの **Inputs** を報告するようになりました。2つの入力は次のとおりです。
- `basis_changes.basisX`（X は整数）: 測定時に適用される、ランダム化ごとの基底回転。これはシンプレクティックにエンコードされています — `I=0, Z=1, X=2, Y=3`
- `pauli_lindblad_maps.<ref>`: ゲートボックスの学習されたノイズモデル

回路に `ChangeBasis` と `InjectNoise` を追加したことで、samplex が実行時に受け取ると期待するデータが変わりました。`ref` 文字列（一般的な形は `basisX`、`rYYYY`。X は整数、Y は整数または文字）は、そのデータを供給するために使う辞書のキーです。

**Outputs** には `pauli_signs` も加わります。`InjectNoise` があると、各ランダム化はノイズモデルからパウリエラーをサンプリングし、負のレートを持つ因子のパリティが、後で事後処理が期待値に適用する $\pm 1$ の補正になります。

このように、同じパターンがここでも貫かれています。アノテーションが必要なもの（基底変更、ノイズモデル、トワリングなど）を *宣言* し、samplex がその枠を *公開* して、実行時に束縛される結果を報告します。2.5 節では、これらの枠の値を供給し、`Executor` でプログラムを実行します。

## 2.5  `Executor` でジョブを投入する

回路のノイズを学習したので、[__Executor プリミティブ__](https://quantum.cloud.ibm.com/docs/en/guides/get-started-with-executor) を使ってジョブを投入します。`Executor` は Samplomatic のアノテーションを尊重する Runtime プリミティブです。

簡単な復習として、__Sampler__ と __Estimator__ プリミティブは [Primitive Unified Bloc（PUB）](https://quantum.cloud.ibm.com/docs/en/guides/primitive-input-output#pubs) を受け取ることを思い出してください。PUB はタプルで、`QuantumCircuit`、パラメーター、そして Estimator の場合はオブザーバブルが、その入力の一部です。

Executor プリミティブの入力と出力は、Sampler や Estimator プリミティブのものとは大きく異なります。PUB のリストを入力として受け取る代わりに、Executor は `QuantumProgram` を受け取ります。これは `QuantumProgramItem` オブジェクトのリストを含んでいます。これらのコンテナークラスは、単純なタプルデータ構造である PUB よりも柔軟性を持ちます。これらの `QuantumProgramItem` は、次のいずれかになります。
- 回路とそのパラメーター値を格納する `CircuitItem`
- テンプレート回路、samplex、samplex 引数を格納する `SamplexItem`

ここでは、`SamplexItem` を使う Executor ジョブに焦点を当てます。まず、`build` を使ってトイ回路のテンプレートと samplex をビルドします。

In [ ]:
toy_template, toy_samplex = build(boxed_toy)

print(toy_samplex)

次に、samplex 引数を構築します。`samplex_arguments` のキーは、`samplex.inputs()` が返すものと正確に一致しなければなりません。`noise_scales.<ref>` や `basis_changes.<ref>` のような自動生成される ref 名も含みます。

In [ ]:
samplex_arguments = (
    toy_samplex.inputs()
    .make_broadcastable()
    .bind(pauli_lindblad_maps=toy_refs_to_noise)
)

これで `QuantumProgram` を構築できます。`append_samplex_item(...)` メソッドを使って `SamplexItem` を `QuantumProgram` に追加します（複数が共存できます。これがバッチ化された緩和ジョブの作り方です）。

In [ ]:
program = QuantumProgram(shots=64)
program.append_samplex_item(
    toy_template,
    samplex=toy_samplex,
    samplex_arguments=samplex_arguments,
    shape=(16,),
)

これで QPU 上で Executor ジョブを実行する準備ができました。

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `TOY_EXECUTOR_JOB_ID` パラメーターに貼り付け、`SUBMIT_TOY_EXEC_JOB = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは2秒です（ibm_fez でテスト済み）。_ この使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
TOY_EXECUTOR_JOB_ID = None # paste job_id here on re-run
SUBMIT_TOY_EXEC_JOB = False # set True to submit a fresh executor job

executor = Executor(backend)
executor.options.environment.job_tags = ["qgss26"]

if TOY_EXECUTOR_JOB_ID is not None:
    toy_job = service.job(TOY_EXECUTOR_JOB_ID)
    print(f"Re-using saved job: {TOY_EXECUTOR_JOB_ID}")
elif SUBMIT_TOY_EXEC_JOB:
    toy_job = executor.run(program)
    TOY_EXECUTOR_JOB_ID = toy_job.job_id()
    print(f"Submitted Executor job: {TOY_EXECUTOR_JOB_ID}")
else:
    print("Set SUBMIT_TOY_EXEC_JOB=True to submit a fresh executor job, "
          "or paste a saved job id into TOY_EXECUTOR_JOB_ID and re-run.")


下のセルでジョブの状態を確認してください。

In [ ]:
toy_job = service.job(TOY_EXECUTOR_JOB_ID)
print(f"{TOY_EXECUTOR_JOB_ID}  (status: {toy_job.status()})")

結果を取り出し、量子ビット0と量子ビット1それぞれについて $Z$ の期待値を見ます。

In [ ]:
if toy_job.status() == "DONE":
    toy_result = toy_job.result()
    data    = toy_result[0]
    meas    = data["c"]
    flips   = data["measurement_flips.c"]

    # 形状の健全性チェック: レイアウトは SDK のバージョンによって変わりうる。
    print(f"meas shape  : {meas.shape}")
    print(f"flips shape : {flips.shape}")

    corrected = np.bitwise_xor(meas, flips)
    z_vals    = 1 - 2 * corrected

    # 最後の量子ビット軸だけが残るように、先頭のすべての軸（ランダム化、ショット、…）
    # にわたって平均する。これは (randomizations, shots, bits) と (shots, bits) の
    # 形状の違いに対して頑健である。
    z_means = z_vals.reshape(-1, z_vals.shape[-1]).mean(axis=0)

    print(f"\n<Z_0> = {z_means[0]:+.4f}")
    print(f"<Z_1> = {z_means[1]:+.4f}")
else:
    print(f"Not done yet (status={toy_job.status()}). Re-run cell when DONE.")

理想値は両方の量子ビットで $\langle Z_i \rangle = +1$ です。これは、CZ が $|00\rangle$ に自明に作用するため、最終状態は $|00\rangle$ のままであるはずだからです。この状態からのずれはすべてハードウェアノイズです。典型的な実行では、ずれはパーセントレベルで、その大きさはバックエンド、量子ビットのレイアウト、キュー投入時のキャリブレーション、ショット予算によって変わります。

典型的には3つの寄与があります。2つの物理量子ビット上の __読み出しエラー__、単一量子ビットのドレッシング回転中の __アイドルによるデコヒーレンス__、そして CZ からの __残留する確率的パウリエラー__ です。

このずれは、また典型的には2つの量子ビットの間で非対称です。というのも、私たちは `optimization_level=0` でトランスパイルしたからです。量子ビットのペアは忠実度ではなくインデックスで選ばれたため、各量子ビットが異なるエラー予算を持ちます。2.3.3 節の支配的な生成子レート（$10^{-3}$ のオーダー）は桁のオーダーの診断を与えますが、観測されるずれには読み出しエラー、アイドル時間、レイアウト依存のキャリブレーション、蓄積したボックスのオーバーヘッドも含まれるので、レートだけではこの差は決まりません。

2.3.3 節で見たような小さな層ごとのレートは、緩和の効果が見えるようになるには積み重なる必要があります。2.6 節では、同じパイプラインをより深い回路（1次元イジングミラー）に適用し、第3章の PNA と SLC が消費する `refs_to_noise_models` 辞書を生成します。

## 2.6  まとめる: 1次元イジング鎖

これまでは、各 Samplomatic ツールを紹介し、全体のワークフローを理解するために、2量子ビットのトイ CZ 回路を使ってきました。ここからは、1次元横磁場イジング鎖で作業します。これは第3章が続けて扱うのと同じ系です。

ボックス化 → ビルド → ノイズ学習 → 実行 というワークフローは同じで、回路はいまや層ごとのレートが積み重なって測定可能なずれになるほど十分に深くなっています。

この節の最後には、第2章で扱ったすべての知識を試す2つの演習があります。演習2では、`NoiseLearnerV3` に特性評価する層をもっと持たせるために、より深いイジングミラーをボックス化することを求めます。演習3では、結果として得られるプログラム — ボックス化された回路、学習されたノイズ、samplex、`QuantumProgram` — を `Executor` が消費する形にまとめます。

### 2.6.1  ハミルトニアンとトロッター回路

考える系は __1次元横磁場イジング鎖__ で、次のハミルトニアンを持ちます。

$$H \;=\; -J \sum_{\langle i,j\rangle} Z_i Z_j \;+\; h \sum_i X_i.$$

このハミルトニアンは標準的な実用規模のベンチマークです。同じハミルトニアン（2次元 heavy-hex 格子上）が、IBM の 127 量子ビットの実用性実証を駆動しました（[Kim et al., *Evidence for the utility of quantum computing before fault tolerance*, Nature **618**, 500–505 (2023)](https://www.nature.com/articles/s41586-023-06096-3)）。私たちが1次元鎖を使うのは、ブリックワーク構造を検査しやすくするためであり、第3章の PNA と SLC のチュートリアルも同じ系を続けて扱うからです。

下で、ヘルパー関数 `construct_ising_circuit(num_qubits, num_trotter_steps, rx_angle, barrier=True)` を定義します。この関数は、両方の量子ビットに $S^\dagger$ を適用し、続いて $\mathrm{CZ}$ を適用することで各イジング回路を構築します。3つとも対角で交換するので、この列はグローバル位相を除いて $R_{ZZ}(-\pi/2)$ に等しくなります。パラメトリックな `RZZ` としてではなくこのように回路を書くことで、すべての2量子ビットゲートが CZ のままになるので、2.1〜2.5 節のボックス化とドレッシングの機構が直接適用できます。

この回路は、単純な固定角の kicked-Ising 形式のトロッターステップです。横磁場回転は `rx_angle` によって制御され、ZZ 相互作用は固定の CZ ベースのブロックによって実現されます。ここではイジング回路の物理についてこれ以上詳しくは触れません。このラボの焦点はノイズ学習とエラー緩和にあり、イジング回路は単に扱いやすい良い例を提供してくれるものだからです。

In [ ]:
def construct_ising_circuit(num_qubits: int,
                            num_trotter_steps: int,
                            rx_angle: float,
                            barrier: bool = True) -> QuantumCircuit:
    qc = QuantumCircuit(num_qubits)
    for _ in range(num_trotter_steps):
        qc.rx(rx_angle, range(num_qubits))
        if barrier:
            qc.barrier()
        for first_qubit in (1, 2):
            for idx in range(first_qubit, num_qubits, 2):
                qc.sdg([idx - 1, idx])
                qc.cz(idx - 1, idx)
        if barrier:
            qc.barrier()
    return qc

ising = construct_ising_circuit(num_qubits=4, num_trotter_steps=1, rx_angle=np.pi / 8)
ising.draw("mpl", fold=-1)

### 2.6.2  ミラーのトリック

第3章では、1次元イジング鎖回路について _緩和なし_ の結果と _エラー緩和済み_ の結果を比較します。実装したエラー緩和の手法の効果を検証するには、目指すべき理想の結果を知る必要があります。任意の回路については、これには別途の古典シミュレーションが必要で、実用規模ではそのシミュレーション自体が高価、あるいは扱いさえ困難になります。ここでミラーのトリックの出番です。

ミラーのトリックは、対象の回路を取り、その逆を回路の末尾に付け加えます。すなわち、回路を順方向に適用してから逆方向に適用します。ノイズのない理想的な量子コンピューターでは、量子状態は初期状態 $|0\rangle^{\otimes N}$ に戻ります。これの有用な帰結として、高価な古典シミュレーションを一切行わずに、任意のオブザーバブルの理想的な期待値を簡単に計算できます。たとえば、すべての量子ビットで $Z$ の理想的な期待値は 1、すなわちすべての $i$ について $\langle Z_i \rangle = +1$ であることが分かります。測定された $\langle Z_i \rangle$ の $+1$ からのずれはすべてハードウェアノイズによるものです。これが、ミラーをエラー緩和の標準的なベンチマークにしている理由です。

Qiskit では、これは `ising.compose(ising.inverse())` を使って実装され、$U$ に $U^\dagger$ を付け加えます。Qiskit の右から左への規約により、これは $U_\text{mirror} = U^\dagger U = I$ を実現します。

In [ ]:
mirror = ising.compose(ising.inverse())
mirror.measure_all()
mirror_isa = isa_pm.run(mirror)
mirror_isa.draw("mpl", fold=-1, idle_wires=False, scale=0.5)

### 2.6.3  イジングミラーをボックス化する

回路とワークフローはそろいました。次のステップはそれらをまとめることです。

下で、ミラー化された回路を `NoiseLearnerV3` が消費する形に変換します。2.3.1 節の `noise_learning_boxing_pm` をそのまま再利用します。どちらの回路も CZ エンタングリングゲートから構築されているので、同じボックス化戦略が適用でき、各ゲート層は `Twirl` + `InjectNoise` ボックスで包まれることになります。

次に `find_unique_box_instructions(...)` が、ボックス化された回路をその *構造的に相異なる* 層に集約します。等価なボックスは1つのノイズモデルを共有するので、`NoiseLearnerV3` は各ユニークな層を一度だけ特性評価すればよいことを思い出してください。
- `undress_boxes=True` は、ユニークな層を見つけるための比較の前にランダムパウリのドレッシングを取り除きます。ドレッシングはボックス間で異なりますが、基礎となる層は異ならないからです。
- `normalize_annotations=None` は、`Twirl` と `InjectNoise` のボックスアノテーションだけを保ち、それ以外はすべて破棄します

In [ ]:
boxed_circuit_ising = noise_learning_boxing_pm.run(mirror_isa)
unique_layers_ising = find_unique_box_instructions(
    boxed_circuit_ising,
    normalize_annotations=None,
    undress_boxes=True,
)

print(f"Unique 2Q-gate layers: {len(unique_layers_ising)}")
for i, inst in enumerate(unique_layers_ising):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a is not None else "(none)"
    print(f"  layer {i}: ref = {ref}")

boxed_circuit_ising.draw('mpl', idle_wires=False)


ユニークな層を抽出したので、それらを検査します。その `ref` 文字列（`NoiseLearnerV3` がこれで報告し返します）と、それぞれのゲートの内容です。イジング回路には5つのボックスがあるにもかかわらず、ユニークな層は3つしかなく、そのうちの1つは測定層であることが分かります。

In [ ]:

for i, inst in enumerate(unique_layers_ising):
    a = get_annotation(inst.operation, InjectNoise)
    ref = a.ref if a else "(none)"
    print(f"Layer {i}: ref = {ref}")
    display(inst.operation.body.draw("mpl", fold=-1, idle_wires=False))

### 2.6.4  ノイズを学習する

ボックス化された回路の準備ができたので、そのノイズを学習できます。`unique_layers_ising` を `NoiseLearnerV3` に渡し、結果を `refs_to_noise_models_ising` にまとめます。これは、各層の `InjectNoise.ref` を学習された `PauliLindbladMap` に対応付ける辞書です。両方のゲート層を特性評価したら、それらのノイズプロファイルがどれほど異なるかを読み取れます。

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `NOISE_LEARN_JOB_ID_ISING` パラメーターに貼り付け、`SUBMIT_NOISE_JOB_ISING = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは10秒です（ibm_fez でテスト済み）。_ この使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
NOISE_LEARN_JOB_ID_ISING = None # paste job_id here on re-run
SUBMIT_NOISE_JOB_ISING   = False   # set True to submit a fresh learning job
learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_ISING is not None:
    learner_job_ising = service.job(NOISE_LEARN_JOB_ID_ISING)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_ISING}")
elif SUBMIT_NOISE_JOB_ISING:
    learner_job_ising = learner.run(unique_layers_ising)
    NOISE_LEARN_JOB_ID_ISING = learner_job_ising.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_ISING}")
else:
    print("Set SUBMIT_NOISE_JOB_ISING=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_ISING and re-run.")


In [ ]:
learner_job_ising = service.job(NOISE_LEARN_JOB_ID_ISING)
print(f"{NOISE_LEARN_JOB_ID_ISING}  (status: {learner_job_ising.status()})")

In [ ]:
if learner_job_ising.status() == "DONE":
    noise_learner_result_ising = learner_job_ising.result()
else:
    print(f"Not done yet (status={learner_job_ising.status()}). Re-run cell when DONE.")

In [ ]:
if 'noise_learner_result_ising' in dir() and noise_learner_result_ising is not None:
    refs_to_noise_models_ising = noise_learner_result_ising.to_dict(unique_layers_ising, require_refs=False)
    print(f"refs_to_noise_models_ising has {len(refs_to_noise_models_ising)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


辞書の準備ができました。各エントリーは1つの学習された層です。何が学習されたかを見るために、下のセルは2つの層の特性を出力します。各層について、生成子の数、そのうち非ゼロのレートを持つものの数、最大の生成子レート、層内のすべてのレートの和（総ノイズ）、そしてどのパウリ項が最も寄与しているかを出力します。

In [ ]:
if "refs_to_noise_models_ising" not in globals() or not refs_to_noise_models_ising:
    print("Fetch a completed Ising noise-learning result first.")
else:
    print("Layer comparison:\n")
    print(f"  {'ref':<10}{'#gens':>8}{'#nonzero':>10}{'max rate':>14}{'sum rates':>14}{'top generator':>22}")
    print("  " + "─" * 78)
    for ref, plm in refs_to_noise_models_ising.items():
        gens = plm.to_sparse_list()
        rates = [abs(r) for _, _, r in gens]
        nonzero = [r for r in rates if r > 0]
        top = max(gens, key=lambda g: abs(g[2]))
        top_label = f"{top[0]}@{tuple(top[1])} ({top[2]:.2e})"
        print(f"  {ref:<10}{len(gens):>8}{len(nonzero):>10}"
              f"{max(rates):>14.4e}{sum(rates):>14.4e}"
              f"{top_label:>22}")


2つのユニークな層はブリックワークの中で似た役割を果たしますが — 偶数のボンド（ここでは2つの CZ ゲートを持つ層）と奇数のボンド（ここでは CZ ゲートが1つだけの層）を実装します。上の表は、それらの学習されたノイズが異なることを示しています。

`sum rates` 列は各層の総ノイズを与え、`top generator` 列はどのチャネルが支配的かを示します。一方の層の支配的なチャネルが、もう一方では弱いか存在しないことがあります。`#nonzero` 列は、一方の層がより少ないチャネルにノイズを集中させ、もう一方はより多くのチャネルに広げていることを示します。

この層ごとのノイズの違いこそが、エラー緩和を層ごとに実装したい理由です。ある層の支配的な生成子に合わせて調整された一様な補正は、もう一方の層のそれを見逃したり、過剰に補正したりします。これはまさに、Runtime の回路全体の `EstimatorOptions`（1.1 節）が埋められないギャップです。

第3章では、PNA が各層の *自身の* 逆チャネルをノイズ緩和オブザーバブルに吸収する方法と、SLC が各層をそれ自身のライトコーンに沿ってスケーリングする方法を見ます。どちらの手法も、いま出力した層ごとのレートを読み取ります。

_注意:_ あなたの具体的な結果は、使っている QPU バックエンドと、その QPU から学習されたノイズによって変わります。ノイズのドリフトとキャリブレーションは毎日起こります。同じ QPU でも、ノイズは時間とともに変化します。

### 演習2 — より深いイジングミラーをボックス化する

<div class="alert alert-block alert-success">

2.6.1〜2.6.4 節では、4量子ビット、1トロッターステップのイジング回路について、完全なノイズ学習ワークフローを一通り見ました。

同じワークフローを、より深い回路に対して自分で実行してください。演習3では、ここで作ったものを取り上げて `Executor` プログラムにまとめます。

- **回路:** 6量子ビットのイジング鎖、2トロッターステップ、`rx_angle = π/8`、`barrier=False`
- **ミラー:** `ising_ex2` を合成し、`barrier()` を追加し、`ising_ex2.inverse()` を合成し、その後 `measure_all()`
- **トランスパイル:** `isa_pm` を通す
- **ボックス化:** `noise_learning_boxing_pm` を通す

順方向と逆方向の半分の間の `barrier()` は、トランスパイラーの最適化パスがミラーを恒等演算に打ち消すのを防ぎます。

下のセルを埋めて、`boxed_circuit_ex2` が、`build()` が受け入れる、Twirl + InjectNoise アノテーション付きのボックス化されたミラー回路になるようにしてください。

</div>

In [ ]:
ising_ex2 = None
mirror_ex2 = None
boxed_circuit_ex2 = None

#TODO: 下にコードを書く。

# 1. 6量子ビット、2ステップのイジング回路を構築する（rx_angle = π/8、barrier=False）
ising_ex2 = ...

# 2. mirror_ex2 を構築する: 順方向 + barrier + 逆方向 + measure_all
#    （barrier はトランスパイラーがミラーを打ち消すのを防ぐ。）
mirror_ex2 = QuantumCircuit(num_qubits)
mirror_ex2.compose(..., inplace=True)   # 順方向
mirror_ex2.barrier()
mirror_ex2.compose(..., inplace=True)   # 逆方向
mirror_ex2.measure_all()

# 3. isa_pm を通してトランスパイルし、noise_learning_boxing_pm を通してボックス化する
mirror_ex2_isa = ...
boxed_circuit_ex2 = ...

In [ ]:
# 解答を採点する
grade_lab3_ex2(ising_ex2, mirror_ex2, boxed_circuit_ex2)

In [ ]:
# すべてのラボにわたる進捗を確認する
check_progress()

演習2では、`boxed_circuit_ex2`、すなわち各ゲート層に `InjectNoise` 枠を持つボックス化されたイジングミラー回路を作りました。このボックス化された回路は、特性評価の準備ができました。

演習3でそれを実行可能なプログラムにまとめる前に、それらの枠を埋めます。これは 2.6.4 節と同じ `NoiseLearnerV3` のワークフローを、いまはより深い回路に対して行うものです。まったく同じ方法で実行します。ジョブを投入し、その状態をポーリングし、その後結果を取得します。学習されたノイズから、`refs_to_noise_models_ex3`、すなわち演習3に必要な `{ref: PauliLindbladMap}` 辞書を生成します。

#### ノイズを学習する（演習3に必要）:

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `NOISE_LEARN_JOB_ID_EX3` パラメーターに貼り付け、`SUBMIT_NOISE_JOB_EX3 = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは4秒です（ibm_fez でテスト済み）。_ この使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
unique_layers_ex3 = find_unique_box_instructions(
    boxed_circuit_ex2,
    normalize_annotations=None,
    undress_boxes=True,
)

NOISE_LEARN_JOB_ID_EX3 = None   # paste a saved job id to re-fetch
SUBMIT_NOISE_JOB_EX3   = False              # set True to submit a fresh job
learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_EX3 is not None:
    learner_job_ex3 = service.job(NOISE_LEARN_JOB_ID_EX3)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_EX3}")
elif SUBMIT_NOISE_JOB_EX3:
    learner_job_ex3 = learner.run(unique_layers_ex3)
    NOISE_LEARN_JOB_ID_EX3 = learner_job_ex3.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_EX3}")
else:
    print("Set SUBMIT_NOISE_JOB_EX3=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_EX3 and re-run.")

In [ ]:
learner_job_ex3 = service.job(NOISE_LEARN_JOB_ID_EX3)
status = learner_job_ex3.status()

print(f"job_id : {NOISE_LEARN_JOB_ID_EX3}")
print(f"status : {status}")

if status == "DONE":
    print("\n  Ready — proceed to Exercise 3.")

ジョブが `DONE` になったら、下のセルを実行して結果を取得し、それを `refs_to_noise_models_ex3`、すなわち層ごとに1つの学習されたモデルを持つ `{ref: PauliLindbladMap}` 辞書に集めてください。このオブジェクトは演習3に必要です。

In [ ]:
noise_result_ex3 = learner_job_ex3.result()
refs_to_noise_models_ex3 = noise_result_ex3.to_dict(
    unique_layers_ex3, require_refs=False
)
print(f"Learned noise for {len(refs_to_noise_models_ex3)} layers: "
      f"{list(refs_to_noise_models_ex3.keys())}")

学習されたノイズモデルは、生のレートよりも図として見たほうが分かりやすく理解できます。下のプロットは、各層の最大の15個のノイズ生成子を示し、1体（単一量子ビット）と2体（2量子ビット）の項を別々に色分けしています。どの生成子が各層のノイズモデルを支配しているかがはっきり分かります。

In [ ]:
# 層ごとに上位15個のノイズ生成子をプロットする
fig, axes = plt.subplots(1, len(refs_to_noise_models_ex3), figsize=(12, 4))
if len(refs_to_noise_models_ex3) == 1: axes = [axes]

for ax, (ref, plm) in zip(axes, refs_to_noise_models_ex3.items()):
    gens = sorted(plm.to_sparse_list(), key=lambda g: -abs(g[2]))[:15]
    labels = [f"{p}@{tuple(q)}" for p, q, _ in gens]
    rates  = [r for _, _, r in gens]
    colors = ['#2c7fb8' if len(p) == 1 else '#d95f0e' for p, _, _ in gens]
    ax.barh(range(len(labels)), rates, color=colors)
    ax.set_yticks(range(len(labels)))
    ax.set_yticklabels(labels, fontsize=8)
    ax.invert_yaxis()
    ax.set_xlabel("Rate")
    ax.set_title(f"Layer {ref}")
    ax.grid(axis="x", alpha=0.3)

fig.legend(handles=[Patch(facecolor="#2c7fb8", label="1-qubit"),
                    Patch(facecolor="#d95f0e", label="2-qubit")],
           loc="upper right", fontsize=9)
fig.suptitle("Top 15 noise generators per layer")
plt.tight_layout()
plt.show()

これでノイズモデル、特に辞書 `noise_result_ex3` が手に入ったので、Executor ジョブを実行できます。

演習3では、`noise_result_ex3` と `build()` 関数を使い、演習2で用意したボックス化された回路を使ってテンプレートと samplex を用意し、Executor ジョブを準備することを求めます。

### 演習3 — `Executor` プログラムをまとめる

<div class="alert alert-block alert-success">

上のノイズ学習ジョブは、`boxed_circuit_ex2` の層ごとに1つのノイズモデルを持つ `{ref: PauliLindbladMap}` 辞書 `refs_to_noise_models_ex3` を学習しました。

この演習では、そのノイズモデルを `Executor` が実行するデータ構造に差し込みます。第2章で紹介した3つのツールすべてを一度に使います。**Samplomatic** の `build`、いま作った **NoiseLearnerV3** の結果、そして **Executor** の `QuantumProgram` です。

プログラムを4ステップで構築します。

- **ビルド:** `build(boxed_circuit_ex2)` を呼んで `template_ex3` と `samplex_ex3` を得る
- **検査:** `samplex_ex3.inputs()` は samplex が期待する `pauli_lindblad_maps.<ref>` 枠を報告します。これらの ref は `refs_to_noise_models_ex3` のキーと一致しなければなりません
- **束縛:** `samplex_ex3.inputs()` から、`make_broadcastable()` を呼び、続いて学習された辞書で `bind(pauli_lindblad_maps=...)` を呼ぶ → `samplex_args_ex3`
- **組み立て:** `QuantumProgram(shots=64)` を作り、テンプレート、samplex、引数を `append_samplex_item(...)` する → `program_ex3`

grader は、プログラムが正しく組み立てられているかをチェックします — ハードウェア上では実行しません。あとに続くオプションのセルで、このプログラムが実際のバックエンド実行から返す期待値を示します。

</div>

In [ ]:
#TODO: ここにコードを追加し、欠けている値を設定する。

template_ex3      = None
samplex_ex3       = None
samplex_args_ex3  = None
program_ex3       = None


# 1. boxed_circuit_ex2 からテンプレートと samplex をビルドする
template_ex3, samplex_ex3 = ...

# 2. samplex が何を期待しているか検査する（pauli_lindblad_maps.<ref> 枠を見る）
print(samplex_ex3.inputs())

# 3. 学習されたノイズモデルを samplex の入力に束縛する。
#    samplex_ex3.inputs() から始め、broadcastable にしてから、
#    refs_to_noise_models_ex3 辞書を `pauli_lindblad_maps` 引数に束縛する。
samplex_args_ex3 = (
    samplex_ex3.inputs()
    .make_broadcastable()
    .bind(pauli_lindblad_maps=...)
)

# 4. 単一の samplex item で QuantumProgram を組み立てる
program_ex3 = QuantumProgram(shots=64)
program_ex3.append_samplex_item(...)

In [ ]:
# 解答を採点する
grade_lab3_ex3(
    template_ex3,
    samplex_ex3,
    samplex_args_ex3,
    program_ex3,
    refs_to_noise_models_ex3,
)

In [ ]:
# すべてのラボにわたる進捗を確認する
check_progress()

#### （オプション）ハードウェアでプログラムを実行する

このステップはオプションで、第3章に進むために必須ではありません。`program_ex3` を実際にハードウェアで実行したときに何が返されるかを示します。この節は、トイ回路について `Executor` ジョブを投入した 2.5 節と同じパターンに従います。

ここでの回路はイジングミラー回路なので、理想の結果はすべての量子ビットで $\langle Z_i\rangle = +1$ です。これから見えるずれはすべて、ボックス化して学習したパイプラインがまだ残しているノイズによるものです。第3章では、結果を改善するためにエラー緩和の手法を実装します。

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `EXECUTOR_JOB_ID_EX3` パラメーターに貼り付け、`SUBMIT_EXECUTOR_JOB_EX3 = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは2秒です（ibm_pittsburgh でテスト済み）。_ 上の使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
executor = Executor(backend)

EXECUTOR_JOB_ID_EX3      = None      # paste an Executor job_id here on re-run
SUBMIT_EXECUTOR_JOB_EX3  = False     # set True to submit a fresh Executor job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_EX3 is not None:
    exec_job_ex3 = service.job(EXECUTOR_JOB_ID_EX3)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_EX3}")
elif SUBMIT_EXECUTOR_JOB_EX3:
    exec_job_ex3 = executor.run(program_ex3)
    EXECUTOR_JOB_ID_EX3 = exec_job_ex3.job_id()
    print(f"Submitted Executor job: {EXECUTOR_JOB_ID_EX3}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_EX3=True to submit a fresh Executor job, "
          "or paste a saved job id into EXECUTOR_JOB_ID_EX3 and re-run.")

In [ ]:
exec_job_ex3 = service.job(EXECUTOR_JOB_ID_EX3)
print(f"{EXECUTOR_JOB_ID_EX3}  (status: {exec_job_ex3.status()})")

結果をプロットします。

In [ ]:
if exec_job_ex3.status() == "DONE":
    exec_result_ex3 = exec_job_ex3.result()
    data  = exec_result_ex3[0]
    meas  = data["meas"]
    flips = data["measurement_flips.meas"]

    corrected = np.bitwise_xor(meas, flips)
    z_vals    = 1 - 2 * corrected
    z_means   = z_vals.reshape(-1, z_vals.shape[-1]).mean(axis=0)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(range(len(z_means)), z_means, color="#2c7fb8")
    ax.axhline(1.0, color="gray", ls="--", lw=1, label="ideal ⟨Z⟩ = +1")
    ax.set_xlabel("Qubit")
    ax.set_ylabel("⟨Z⟩")
    ax.set_ylim(0, 1.05)
    ax.set_title("Mirror expectation values (boxed + learned, pre-mitigation)")
    ax.legend()
    plt.tight_layout()
    plt.show()

    print(f"mean ⟨Z⟩ = {z_means.mean():+.4f}  (ideal = +1)")
else:
    print(f"Not done yet (status={exec_job_ex3.status()}). Re-run cell when DONE.")

#### 第2章のまとめ:

おめでとうございます！Lab 3 の第2章を完了しました。この章では、第3章の高度なエラー緩和の手法を実装するために必要なツールをすべて紹介しました。

Samplomatic + ノイズ学習のワークフロー全体を、まず単純な2量子ビットのトイモデルについて、次により複雑な4量子ビットのイジング鎖について、最後に6量子ビットのイジング鎖について、一通り見ました。どの回路についても、ワークフローは同じままで、変わったのは回路だけでした。

- __演習2__ では、6量子ビットのイジングミラー回路をボックス化し、ノイズ学習ステップがその層を特性評価しました。
- __演習3__ では、ボックス化された回路、以前に学習したノイズ、samplex を、`Executor` が実行する `QuantumProgram` にまとめました。

その過程で、層ごとのノイズ構造を検査しました。`InjectNoise` アノテーションを持つすべての層に付けられた `ref` 文字列が、学習されたノイズ情報を結びつけるラベルであることを学びました。

イジングミラー回路については、層が回路の物理の中で似た役割を果たすにもかかわらず、各層で学習されたノイズモデルが異なる（特に層ごとに支配的な生成子が異なる）ことを見ました。これはまさに、エラー緩和を __層ごとに__ 作用させたい理由を動機付けます。

#### 何が欠けているか — そしてなぜ第3章があるのか:

重要なことに、第2章で見たパイプラインはノイズを測定して実行の準備をしますが、それを除去はしません。ここまで、トワリングがコヒーレントエラーを確率的なパウリチャネルに整形するものの、それを除去はしないことを見ました（有限個のランダム化サンプルを適用するだけで、逆ノイズを適用することは決してありません）。また、学習されたノイズ辞書は層ごとのノイズを記述しますが、やはり第2章はその逆を決して使いません。`inject_noise_strategy="no_modification"` のもとでは、辞書は素通し（passthrough）であり、それがイジングミラー回路の $\langle Z_i\rangle$ の結果が理想値 $+1$ を下回る理由です。回路が深くなるにつれて、層ごとのレートが積み重なり、緩和なしのバイアスによって結果が信頼できなくなる点に達します。そこで層ごとのエラー緩和が必要になります。

第3章では、次のステップのための手法を紹介します。そこでは `inject_noise_strategy` を変える必要があります。
- PNA（`inject_noise_strategy=uniform_modification`）は、学習されたチャネルの逆を _ノイズ緩和オブザーバブル_ に吸収します
- SLC（`inject_noise_strategy=individual_modification`）は、各オブザーバブルの _ライトコーン_ に沿って反ノイズをサンプリングします

## 第3章 — エラー緩和のための Qiskit アドオン

第3章へようこそ！この章では、第2章で紹介したツールを使って、高度な _層ごとの_ エラー緩和の手法を実装します。ここまでの流れを振り返りましょう。第1章では、`Estimator` のオプションとして使えるエラー緩和の手法（`DD`、`PT`、`TREX`、`ZNE`、`PEA`、`PEC`）を見直しました。これらは _回路全体のスイッチ_ です（`PEC` と `PEA` は *内部的には* 層ごとの緩和を行いますが、API は層ごとのつまみを表に出しません）。第2章では Samplomatic を紹介し、回路をボックス化してエラー緩和の判断を特定の層に局所化できるようにしました。第2章ではノイズ学習も見せました。

この章では、それらの要素を、回路全体の技術を超える構造を意識したアドオンに組み合わせます。3つの手法（2つの Qiskit アドオン）に焦点を当てます。

- **3.1 節: Propagated noise absorption（PNA）** — オブザーバブルを書き換えることでゲートノイズを緩和し、回路そのものは変えないままにします。
- **3.2 節: Probabilistic error cancellation（PEC）** — 回路の逆ノイズ書き換えからサンプリングすることでエラーを緩和します。
- **3.2 節: Shaded lightcones（SLC）** — PEC の考えを選択的に適用し、オブザーバブルに影響しうるものだけを緩和してサンプリングオーバーヘッドを減らします。

_PNA_ と _SLC_ がこのラボの主な焦点です。これらは、このラボのタイトルが指す *量子優位性のための新しいツール* です。層ごとに作用し、各層の *自身の* 学習されたノイズを吸収または打ち消すことで、これらは Runtime `Estimator` オプションができることを超え、回路が大きくなっても実用的でいられるほどサンプリングオーバーヘッドを低く保ちます。

第2章と同じ1次元横磁場イジングミラー回路を使ってエラー緩和の手法を探ります。2.6.1 節の `construct_ising_circuit` 関数とミラーのトリックを再利用しますが、層ごとの効果を大きくして、手法を実装したときのその影響を見られるように、より長い鎖へとスケールアップします。

## 3.1 Propagated noise absorption（PNA）

この節では、**[Propagated Noise Absorption（PNA）](https://qiskit.github.io/qiskit-addon-pna/)** を紹介します。これは、学習されたノイズを反転し、それを回路そのものではなくオブザーバブルに吸収するエラー緩和の手法です。従来の probabilistic error cancellation（PEC）が多数のノイズを含む回路のバリアントのサンプリングを必要とする（3.2 節で見ます）のに対して、PNA はそのオーバーヘッドを回避し、第1章と第2章で積み上げてきた構造 — ドレスされた層、クリフォードを通したパウリの伝播、`Twirl` と `InjectNoise` アノテーションを持つボックス化された回路、そして疎なパウリ・リンドブラッド生成子として学習されたノイズ — を活用します。

鍵となる洞察は、ノイズがパウリである（あるいはパウリ形にトワリングできる）とき、その逆は 1.1.3 節のクリフォード共役の規則を使って回路を通して **順方向に伝播** できる、ということです。そのノイズはその後、**測定オブザーバブルに吸収** されて _ノイズ緩和オブザーバブル_ を生み出せます。

パウリチャネルは効率的に合成・伝播するので、PNA は単一のノイズ緩和オブザーバブル $\tilde{O}$ を計算します。これは、ノイズを含む回路で測定したときに、元の望むオブザーバブル $\langle O \rangle$ のノイズのない期待値の推定値を与えます。

この節では、4ステップの PNA ワークフロー — ボックス化、学習、伝播、吸収 — を一通り見ます。第2章のイジングミラー回路を例として使い、それを2トロッターステップの10量子ビットにスケールアップして、この例について PNA ワークフロー全体を実行します。

### 3.1.1 セットアップと10量子ビットへのスケールアップ

この節では、2.6 節で紹介した __1次元横磁場イジング鎖__ ハミルトニアンを引き続き使います。
$$
H = -J \sum_{\langle i,j \rangle} Z_i Z_j + h \sum_i X_i \;.
$$


2トロッターステップと回転角 π/8 を保ったまま、10量子ビットにスケールアップします。下では、単に問題をセットアップします。イジングミラー回路を構築してトランスパイルします。

In [ ]:
# 10量子ビットの設定を作る
# ── 回路 ─────────────────────────────────────────────────────────
num_qubits_pna          = 10           # 鎖の長さ
num_trotter_steps_pna   = 2            # トロッターの「繰り返し数」
rx_angle           = np.pi / 8    # ステップごとの横磁場回転

# イジング鎖回路を作る
ising_pna  = construct_ising_circuit(num_qubits_pna, num_trotter_steps_pna, rx_angle)
mirror_pna = ising_pna.compose(ising_pna.inverse())
mirror_pna.measure_all()

# ISA に従うようにトランスパイルする
mirror_isa_pna = isa_pm.run(mirror_pna)
layout_pna     = mirror_isa_pna.layout.final_index_layout()

print(f"Mirror     : {num_qubits_pna}q × {num_trotter_steps_pna} Trotter steps, "
      f"rx_angle = π/{round(np.pi / rx_angle)}")
print(f"ISA layout : physical qubits {layout_pna}")

### 3.1.2 回路をボックス化してノイズを学習する

新しいイジングミラー回路をボックス化できるようにする前に、PNA 用の新しいボックス化パスマネージャーを作る必要があります。第2章では `inject_noise_strategy="no_modification"` のボックス化パスマネージャーを使いました。これは、学習されたノイズモデルを素通しとして samplex に送り込みますが、サンプリングされた回路には決して適用しませんでした。第3章では異なる戦略が必要です。

- **PNA** は `inject_noise_strategy="uniform_modification"` を使います。すべての層のノイズがサンプリング時に _同じ方法で_ 変更されます。PNA はすべての `ref` で `noise_scales` を 0 に設定します。これによって、各層をその学習されたモデルに関連付けたままにしてノイズモデルをオブザーバブルに伝播できるようにしつつ、サンプリングされた回路には手を加えません。
- **SLC** は `inject_noise_strategy="individual_modification"` を使います。各層（および各生成子）が独立に変更されるので、SLC はオブザーバブルのライトコーンに沿って生成子ごとにノイズをスケーリングできます。これは 3.2 節で見ます。

`inject_noise_strategy` の選択の詳細については、`generate_boxing_pass_manager` の [ドキュメント](https://qiskit.github.io/samplomatic/api/auto/samplomatic.transpiler.generate_boxing_pass_manager.html#samplomatic.transpiler.generate_boxing_pass_manager/inject_noise_strategy) を参照してください。

新しいボックス化パスマネージャーを定義した後、`find_unique_box_instructions` を使って回路のユニークな層を見つけます。下でユニークな層の数を出力します。

In [ ]:
# ═══════════════════════════════════════════════════════════════════
# PNA — uniform_modification 用のボックス化マネージャー + ユニークな層を見つける
# ═══════════════════════════════════════════════════════════════════
pna_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="uniform_modification",
)

# PNA 用に回路をボックス化する
boxed_circuit_pna = pna_boxing_pm.run(mirror_isa_pna)
# 回路のユニークな層を見つける
unique_layers_pna = find_unique_box_instructions(
    boxed_circuit_pna, normalize_annotations=None, undress_boxes=True,
)
print(f"PNA boxed  : {len(unique_layers_pna)} unique layers")

これで回路のノイズを学習する準備ができました。この10量子ビットのイジング回路については、以前ノイズを学習した回路とは異なる回路なので、ここで改めて学習する必要があります。量子コンピューターのノイズは時間とともに変化する（キャリブレーションは毎日行われます）ので、以前のキャリブレーションウィンドウでキャッシュされた学習結果は、一般に学習し直すべきです。

まず、ノイズ学習のオプションを初期化します。

In [ ]:
# ── NoiseLearnerV3 の新しいオプション（PNA、SLC が使い、PEC も間接的に使う） ───────
NL_NUM_RANDOMIZATIONS  = 16
NL_SHOTS_PER_RAND      = 32
NL_LAYER_PAIR_DEPTHS   = [1, 2, 4, 8]

pna_nl_options = NoiseLearnerV3Options(
    num_randomizations=NL_NUM_RANDOMIZATIONS,
    shots_per_randomization=NL_SHOTS_PER_RAND,
    layer_pair_depths=NL_LAYER_PAIR_DEPTHS,
)
pna_learner = NoiseLearnerV3(backend, pna_nl_options)

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `NOISE_LEARN_JOB_ID_PNA` パラメーターに貼り付け、`SUBMIT_NOISE_JOB_PNA = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは15秒です（ibm_pittsburgh でテスト済み）。_ 上の使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
NOISE_LEARN_JOB_ID_PNA = None # paste job_id here on re-run
SUBMIT_NOISE_JOB_PNA   = False   # set True to submit a fresh learning job
pna_learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_PNA is not None:
    learner_job_pna = service.job(NOISE_LEARN_JOB_ID_PNA)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_PNA}")
elif SUBMIT_NOISE_JOB_PNA:
    learner_job_pna = pna_learner.run(unique_layers_pna)
    NOISE_LEARN_JOB_ID_PNA = learner_job_pna.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_PNA}")
else:
    print("Set SUBMIT_NOISE_JOB_PNA=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_PNA and re-run.")


In [ ]:
learner_job_pna = service.job(NOISE_LEARN_JOB_ID_PNA)
print(f"{NOISE_LEARN_JOB_ID_PNA}  (status: {learner_job_pna.status()})")

In [ ]:
if learner_job_pna.status() == "DONE":
    noise_learner_result_pna = learner_job_pna.result()
else:
    print(f"Not done yet (status={learner_job_pna.status()}). Re-run cell when DONE.")

各ユニークな層で学習されたノイズを含む辞書 `refs_to_noise_models_pna` を取り出します。

In [ ]:
if 'noise_learner_result_pna' in dir() and noise_learner_result_pna is not None:
    refs_to_noise_models_pna = noise_learner_result_pna.to_dict(unique_layers_pna, require_refs=False)
    print(f"refs_to_noise_models_pna has {len(refs_to_noise_models_pna)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


### 対象のオブザーバブル

横磁場イジングハミルトニアンの一部は ZZ 相互作用エネルギー項です。対象のオブザーバブルとして、そのような項の1つを、真ん中の2つの量子ビット上で考えます。

$$
O \;=\;  Z_{4} Z_{5}, \;
$$
$N=10$ 量子ビットについてです。これは最近接の ZZ 相関です。ミラー回路の理想出力 $|0\rangle^{\otimes N}$ では、すべての $\langle Z_i Z_{i+1}\rangle = +1$ なので、理想的な期待値は $\langle O\rangle = 1$ です。

PNA は、オブザーバブル $O$ を修正された _ノイズ緩和_ オブザーバブル $\tilde{O}$ に書き換えます。この修正されたオブザーバブルには、異なる係数を持つ追加の項が含まれます。考え方は、$\tilde{O}$ のノイズを含む期待値が、元の $O$ の理想的な期待値に一致する、というものです。

まず、オブザーバブル $O$ を構築し、その後それに回路のレイアウトを適用します。

In [ ]:
# イジングモデルのオブザーバブルを作る
# 真ん中の2つの量子ビット上の単一の ZZ（1項、理想の期待値 = 1）。
mid = num_qubits_pna // 2
observable_pna =  SparsePauliOp.from_sparse_list(
    [("ZZ", [mid - 1, mid], 1.0)], num_qubits=num_qubits_pna)

print(observable_pna)

In [ ]:
# オブザーバブルにレイアウトを適用する
observable_pna_isa = observable_pna.apply_layout(mirror_isa_pna.layout)

### 3.1.3  ノイズ緩和オブザーバブル $\tilde{O}$ を計算する

オブザーバブル $O$ を構築したので、PNA ワークフローの次のステップは、PNA アドオンの関数 [`generate_noise_mitigating_observable`](https://qiskit.github.io/qiskit-addon-pna/apidocs/qiskit_addon_pna.html#qiskit_addon_pna.generate_noise_mitigating_observable) を使ってノイズ緩和オブザーバブル $\tilde{O}$ を計算することです。この関数は、ボックス化された回路、オブザーバブル、学習されたノイズ辞書を受け取ります。

各 `InjectNoise(ref=...)` アノテーションは、ノイズ学習ステップからその層の `PauliLindbladMap` を知っています。PNA はそれらのチャネルの逆を回路を通して伝播させ、オブザーバブルに折り込みます。結果は、ノイズを含む期待値が元の $O$ の理想的な期待値に等しくなる新しいオブザーバブル $\tilde{O}$ です。

切り捨てステップが支配的な項だけを保つので、$\tilde{O}$ は測定可能なままです。オブザーバブルの項が多いほど、その期待値を推定するのに必要な回路の測定が多くなります。切り捨ては `max_err_terms` と `max_obs_terms` で制御されます。

ボックス化された回路そのものは変わりません。変わるのはオブザーバブルだけです。

In [ ]:
# PNA のパラメーター
num_processes = 8
max_err_terms = 10000
max_obs_terms = 10000

# ノイズ緩和オブザーバブルを生成する
obs_tilde_isa = generate_noise_mitigating_observable(
    boxed_circuit_pna,
    observable_pna_isa,
    refs_to_noise_models_pna,
    max_err_terms=max_err_terms,
    max_obs_terms=max_obs_terms,
    num_processes=num_processes,
    print_progress=True,
    search_step=8,
)

In [ ]:
print(f"Number of Pauli terms in the noise mitigating observable: {len(obs_tilde_isa)}")

ノイズ緩和オブザーバブル $\tilde{O}$ を論理量子ビット空間で構築するために、いくつかの事後処理を行う必要があります。これは、オブザーバブルを検査してどんな項を含んでいるかを見られるようにするのに役立ちます。

In [ ]:
### ノイズ緩和オブザーバブルを物理量子ビットから論理量子ビットへマップする

# 逆レイアウトを作る: 物理量子ビットのインデックス → 論理量子ビットのインデックス
physical_to_logical = {
    physical_q: logical_q
    for logical_q, physical_q in enumerate(layout_pna)
}

# ノイズ緩和オブザーバブルを疎な (Pauli, qubits, coeff) 形式で取り出す
isa_pauli_terms = obs_tilde_isa.to_sparse_list()

# 各パウリ項を物理量子ビットから論理量子ビットへ再マップする
logical_pauli_terms = [
    (
        pauli_string,                                  # パウリ演算子（例 "ZIX"）
        [physical_to_logical[q] for q in phys_qubits], # 物理 → 論理 のインデックス
        coefficient                                    # 実数値の係数
    )
    for (pauli_string, phys_qubits, coefficient) in isa_pauli_terms
]

# 論理（仮想）量子ビット空間でオブザーバブルを再構築する
obs_tilde_virtual = SparsePauliOp.from_sparse_list(
    logical_pauli_terms,
    num_qubits=num_qubits_pna
)

下で、$\tilde{O}$ のすべての項の大きさの分布をプロットします。

In [ ]:
# プロットのためにオブザーバブルの大きさを降順にソートする
obs_tilde_virtual = obs_tilde_virtual[np.argsort(np.abs(obs_tilde_virtual.coeffs))][::-1]

plt.figure(figsize=(8, 5))
plt.plot(np.abs(obs_tilde_virtual.coeffs), "o", color="tab:blue", markersize=6, alpha=0.7)
plt.yscale("log")
plt.xlabel("Pauli term index", fontsize=14)
plt.ylabel("Magnitude", fontsize=14)
plt.title(r"$\tilde{O}$ coefficient magnitudes", fontsize=14)
plt.grid(axis="both", alpha=0.3)
plt.tight_layout()
plt.show()

$\tilde{O}$ には多くの項があり、その多くが非常に小さい大きさであることが分かります。大きさでソートして、構造を検査するために最初の10項を示しましょう。元のオブザーバブルは1項だけ（中央の2つの量子ビット上の ZZ）だったことを思い出してください。

In [ ]:
### ノイズ緩和オブザーバブルを切り捨てる
num_terms_to_plot = 10

# すべての係数の絶対値を取る
coeff_magnitudes = np.abs(obs_tilde_virtual.coeffs)

# 係数を小さい順から大きい順にソートするインデックスを得る
sorted_indices = np.argsort(coeff_magnitudes)

# 最大の係数を先頭に置くために順序を反転し、ソートされたインデックスで並べ替える
sorted_indices_desc = sorted_indices[::-1]
obs_tilde_virtual = obs_tilde_virtual[sorted_indices_desc]

# 最も重要なパウリ項だけを保つ
obs_tilde_virtual = obs_tilde_virtual[:num_terms_to_plot]

In [ ]:
def full_pauli_string(pstr, qubits, n):
    full = ["I"] * n
    for p, q in zip(pstr, qubits):
        full[q] = p
    return "".join(reversed(full))   # 量子ビット0を左端に

# |係数| の降順でソートし、上位10個を保つ。
sorted_idx = np.argsort(np.abs(obs_tilde_virtual.coeffs))[::-1][:10]
obs_tilde_top = obs_tilde_virtual[sorted_idx]

# 上位10項のうち、どれが元の項でどれが PNA が追加した項かを識別する。
target_paulis_virtual = {
    full_pauli_string(p, q, obs_tilde_virtual.num_qubits)
    for p, q, _ in observable_pna.to_sparse_list()
}
labels = [
    full_pauli_string(p, q, obs_tilde_virtual.num_qubits)
    for p, q, _ in obs_tilde_top.to_sparse_list()
]
coeffs = np.abs(obs_tilde_top.coeffs)
is_orig = np.array([l in target_paulis_virtual for l in labels])

# プロットする。
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(labels))
ax.plot(idx[is_orig],  coeffs[is_orig],  "s", color="C3", markersize=12, label="Original ZZ term")
ax.plot(idx[~is_orig], coeffs[~is_orig], "o", color="C0", markersize=10, label="PNA-added")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha="right", family="monospace")
ax.set_xlabel("Pauli term (virtual qubits)")
ax.set_ylabel(r"$|\tilde{c}|$")
ax.set_title(r"Top 10 terms in $\tilde{O}_{Z}$")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

元の ZZ 項が最大の大きさを持つのが見えるはずです。追加の項は回路のノイズを説明するものなので、大きさははるかに小さいと期待されます。

元のオブザーバブル $O$ とノイズ緩和オブザーバブル $\tilde{O}$ の項の大きさを直接比較しましょう。

In [ ]:
# パウリ文字列と係数を取り出す
pauli_strings = observable_pna.paulis.to_labels()
coeffs = observable_pna.coeffs

# 係数の大きさでソートする
sorted_indices = sorted(
    range(len(coeffs)),
    key=lambda i: abs(coeffs[i]),
    reverse=True
)

# ヘッダーを出力する
print("\n Original Observable Pauli Terms:\n")
print(f"{'Pauli string':<{observable_pna.num_qubits + 2}}  Coefficient")
print("-" * (observable_pna.num_qubits + 20))

# 項を出力する 
for i in sorted_indices[:num_terms_to_plot]:
    print(f"{pauli_strings[i]}  {coeffs[i].real:+.3f}")

In [ ]:
num_terms_to_plot = 10 

# パウリ文字列と係数を取り出す
pauli_strings = obs_tilde_virtual.paulis.to_labels()
coeffs = obs_tilde_virtual.coeffs

# 係数の大きさでソートする
sorted_indices = sorted(
    range(len(coeffs)),
    key=lambda i: abs(coeffs[i]),
    reverse=True
)

# ヘッダーを出力する
print("\n Noise mitigating Observable Pauli Terms (top 10):\n")
print(f"{'Pauli string':<{obs_tilde_virtual.num_qubits + 2}}  Coefficient")
print("-" * (obs_tilde_virtual.num_qubits + 20))

# 項を出力する（区切り線なし）
for i in sorted_indices[:num_terms_to_plot]:
    print(f"{pauli_strings[i]}  {coeffs[i].real:+.3f}")

中央の ZZ 項は、元のオブザーバブルでは大きさが 1 でした。$\tilde{O}$ ではわずかに 1 を上回り、支配的な項のままです。PNA は元の項を増幅し、層ごとのノイズを吸収するために新しい項を追加しました。上の表は最大の10項を示しています。

あなたの結果は、使っている QPU と、その QPU から学習されたノイズによって変わります。ノイズのドリフトとキャリブレーションは毎日起こります。同じ QPU でも、ノイズは時間とともに変化します。最も正確な結果のためには、最終ジョブを実行する直前に必ずノイズを学習し直すべきです。

### 演習4 — 磁化オブザーバブルを緩和する

<div class="alert alert-block alert-success">

<b>演習4 — 磁化オブザーバブルを緩和する</b>

この演習では、10量子ビットのイジング鎖について、**磁化** オブザーバブルの PNA ノイズ緩和オブザーバブルを構築する必要があります。

$$
O_Z = \sum_{i=0}^{N-1} Z_i, \;
$$

各量子ビット上の $Z$ の和です。ミラーの理想出力 $|0\rangle^{\otimes N}$ ではすべての $\langle Z_i \rangle = +1$ なので、10量子ビットの鎖では理想的な期待値は $\langle O_Z \rangle = 10$ です。

PNA は、各層の学習されたノイズを回路ではなくオブザーバブルに吸収します。ここでは、10量子ビットのボックス化されたミラー上でノイズ緩和磁化オブザーバブル $\tilde{O}_Z$ を構築し、その後、PNA がそこに伝播させた反ノイズ項を見ます。

ノイズ緩和オブザーバブルを3ステップで構築します。

- **ターゲット:** `SparsePauliOp.from_sparse_list` を量子ビットごとに1エントリーで使い、10量子ビット上の $O_Z = \sum_{i=0}^{9} Z_i$ に対する `SparsePauliOp` として `target_observable_ex4` を定義する。
- **レイアウト:** PNA はノイズ層を物理量子ビットのインデックスで照合するので、`apply_layout(mirror_isa_pna.layout)` でオブザーバブルを ISA マップする → `target_observable_ex4_isa`。
- **緩和:** ISA オブザーバブルに対して `generate_noise_mitigating_observable` を呼び、スタブにあらかじめ設定された切り捨てパラメーターをキーワード引数として渡す → `obs_tilde_ex4`。

grader は、ターゲットとノイズ緩和オブザーバブルをチェックします。

</div>

In [ ]:
#TODO: ここにコードを追加し、欠けている値を設定する。


target_observable_ex4     = None
target_observable_ex4_isa = None
obs_tilde_ex4             = None

# PNA のパラメーター（3.1.3 節の本文と同じ）。
num_processes = 8
max_err_terms = 10000
max_obs_terms = 10000

# 1. 各量子ビットに1つずつ、10個の Z 項を持つ SparsePauliOp として
#    target_observable_ex4 を構築する
target_observable_ex4 = ...

# 2. PNA が ISA マップされたオブザーバブルを受け取るように、ターゲットに mirrored_isa_pna.layout を適用する。
target_observable_ex4_isa = ...

# 3. generate_noise_mitigating_observable(boxed_circuit_pna, target_observable_ex4_isa,
#    refs_to_noise_models_pna, max_err_terms=..., max_obs_terms=..., num_processes=...) を呼ぶ。
obs_tilde_ex4 = ...

In [ ]:
# 解答を採点する
grade_lab3_ex4(
    target_observable_ex4,
    target_observable_ex4_isa,
    obs_tilde_ex4
)

In [ ]:
# すべてのラボにわたる進捗を確認する
check_progress()

#### $\tilde{O}_{Z}$ はどんな形か？

下のプロットは、$\tilde{O}_{Z}$ の最大の20項を、仮想量子ビット上のパウリ文字列とともに示しています。元の10項（赤い四角）がわずかに増幅されて上部に位置するのが見えるはずです。青い円は、PNA が学習されたノイズモデルから伝播させた反ノイズ補正です。

In [ ]:
# 読みやすいパウリラベルのために obs_tilde を仮想量子ビットにマップし戻す。
p_2_v = {p: v for v, p in enumerate(layout_pna)}
obs_tilde_virtual_ex4 = SparsePauliOp.from_sparse_list(
    [(p, [p_2_v[q] for q in qs], c) for p, qs, c in obs_tilde_ex4.to_sparse_list()],
    num_qubits=num_qubits_pna,
)

# |係数| の降順でソートし、上位20個を保つ。
sorted_idx = np.argsort(np.abs(obs_tilde_virtual_ex4.coeffs))[::-1][:20]
obs_tilde_top = obs_tilde_virtual_ex4[sorted_idx]

def full_pauli_string(pstr, qubits, n):
    full = ["I"] * n
    for p, q in zip(pstr, qubits):
        full[q] = p
    return "".join(reversed(full))   # 量子ビット0を左端に


# 上位20項のうち、どれが元の項でどれが PNA が追加した項かを識別する。
target_paulis_virtual = {
    full_pauli_string(p, q, num_qubits_pna)
    for p, q, _ in target_observable_ex4.to_sparse_list()
}
labels = [full_pauli_string(p, q, num_qubits_pna)
          for p, q, _ in obs_tilde_top.to_sparse_list()]
coeffs = np.abs(obs_tilde_top.coeffs)
is_orig = np.array([l in target_paulis_virtual for l in labels])

# プロットする。
fig, ax = plt.subplots(figsize=(10, 4))
idx = np.arange(len(labels))
ax.plot(idx[is_orig],  coeffs[is_orig],  "s", color="C3", markersize=12, label="Original observable terms")
ax.plot(idx[~is_orig], coeffs[~is_orig], "o", color="C0", markersize=10, label="PNA-added")
ax.set_yscale("log")
ax.set_xticks(idx)
ax.set_xticklabels(labels, rotation=45, ha="right", family="monospace")
ax.set_xlabel("Pauli term (virtual qubits)")
ax.set_ylabel(r"$|\tilde{c}|$")
ax.set_title(r"Top 20 terms in $\tilde{O}$")
ax.legend()
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

### 3.1.4  ハードウェアで実行する

$\tilde{O}$ の準備ができたので、Executor プログラムを構築して投入します。

この節には次のものが必要です。
- `boxed_circuit_pna`（uniform_modification 付き）
- 演習4の `obs_tilde_ex4`（磁化オブザーバブル用）
- 10量子ビットのノイズ学習からの `refs_to_noise_models_pna`

3.1.5 節では、3つのケースについて期待値の結果を比較します。緩和なしのオブザーバブル、PNA ワークフローからのノイズ緩和オブザーバブル、そして PNA + TREX からのノイズ緩和オブザーバブルです。

In [ ]:
# "Executor" で使うためにテンプレート回路と samplex をビルドする
template_circuit_pna, samplex_pna = samplomatic.build(boxed_circuit_pna)

In [ ]:
# ノイズ緩和オブザーバブルを切り捨てる項の数
num_to_measure = 10

# 正準（測定）順で測定される量子ビットを特定する。
meas_box = boxed_circuit_pna.data[-1]
canonical_qubits = [
    i for i, q in enumerate(boxed_circuit_pna.qubits)
    if q in meas_box.qubits
]

# 量子ビットインデックスの対応付けを構築する: 正準 → 物理 → 仮想（論理）。
canonical_to_physical = dict(enumerate(canonical_qubits))
physical_to_virtual   = {p: v for v, p in enumerate(layout_pna)}
canonical_to_virtual  = {
    c: physical_to_virtual[p]
    for c, p in canonical_to_physical.items()
}

# 下で値をハードコードするのではなく、測定ボックスから ChangeBasis の ref を
# 見つける（samplomatic はボックス化時に ref を自動割り当てし、その値は回路に
# 依存する — 例えば、ある構成では "basis2"、別の構成では "basis0"）。
# この動的な探索によって、この節の残りは回路に依存しないままになる。
change_basis_ann = get_annotation(meas_box.operation, ChangeBasis)
if change_basis_ann is None:
    raise RuntimeError(
        "No ChangeBasis annotation on the measurement box. "
        "The boxing pass manager must be built with measure_annotations='all'."
    )
basis_ref = change_basis_ann.ref
print(f"ChangeBasis ref: {basis_ref!r}")

# 測定前に Õ_Z を最大の項に切り捨てる（ショットあたりの分散が小さくなる）。
obs_tilde_virtual_ex4 = obs_tilde_virtual_ex4[
    np.argsort(np.abs(obs_tilde_virtual_ex4.coeffs))[::-1]
][:num_to_measure]

# 切り捨てた Õ_Z のすべてのパウリ項をカバーするのに必要な基底。
meas_bases, bases_reverser = get_measurement_bases(obs_tilde_virtual_ex4)
meas_bases_canonical = [
    np.array(
        [base[canonical_to_virtual[c]] for c in sorted(canonical_to_virtual)],
        dtype=np.uint8,
    )
    for base in meas_bases
]

# 正準配列の中のエンコード: I=0, Z=1, X=2, Y=3。
# 同じ実行から *緩和なしの* <O_Z> も取り出せるように、常に all-Z 基底を含める。
# 切り捨てた Õ_Z は all-Z を自然には必要としないかもしれない（例えば PNA が
# 追加した X/Y 項が切り捨て後に支配的な場合）が、それを追加してもかかるコストは
# 1つ分の基底のショットだけであり、3.1.5 節を自己完結に保てる。
all_z_canonical = np.ones(num_qubits_pna, dtype=np.uint8)

if any(np.array_equal(b, all_z_canonical) for b in meas_bases_canonical):
    all_z_idx = next(
        i for i, b in enumerate(meas_bases_canonical)
        if np.array_equal(b, all_z_canonical)
    )
    print(f"all-Z basis already present at index {all_z_idx}")
else:
    meas_bases_canonical.append(all_z_canonical)
    all_z_idx = len(meas_bases_canonical) - 1
    print(f"Appended all-Z basis at index {all_z_idx}")

# デコードした基底の出力（量子ビット0を左端に） — 健全性チェックに便利。
def _decode(b):
    return "".join("IZXY"[v] for v in b)

print(f"\nMeasurement bases (canonical order, qubit 0 leftmost):")
for i, b in enumerate(meas_bases_canonical):
    note = "  ← unmitigated baseline" if i == all_z_idx else ""
    print(f"  [{i}] {_decode(b)}{note}")

#### 余談: 項の数 vs. 測定基底の数

上の切り捨ては、固定された *数* の項を保ちますが、ショットコストを左右する量は、相異なる **測定基底** の数です。`get_measurement_bases` は量子ビットごとに交換するパウリ項をグループ化して、それらが基底を共有し、したがってショットを共有するようにします。すでに測定している基底で対角な PNA 追加項は、追加の *基底* コストなしで保てます。

下のセルは、完全な $\tilde{O}_Z$ を再構築し、`num_to_measure` のカットのすぐ先にある項のうち、すでに必要とされている基底の集合に入るものがいくつあるかを数えます。それらを保つのは、測定基底の観点では「ただ」です — もっとも統計的にはまったくただではありません。追加の項はそれぞれ自分の係数とショットノイズを $\langle\tilde{O}_Z\rangle$ の総和推定値に寄与します。節約になるのは、新しい基底、したがって新しいショットの割り当てが不要になる点です。

In [ ]:
# ── 余談: 項の数 vs. 測定基底の数 ───────────────────────────────
#
# *完全な* O_Z-tilde を再構築する（obs_tilde_virtual_ex4 は上でその場で切り捨てられた）。
# こうすることでカットの先までスキャンして、どの追加の項が「ただ」かを見られる。
p_2_v = {p: v for v, p in enumerate(layout_pna)}
obs_tilde_virtual_full = SparsePauliOp.from_sparse_list(
    [(p, [p_2_v[q] for q in qs], c) for p, qs, c in obs_tilde_ex4.to_sparse_list()],
    num_qubits=num_qubits_pna,
)
obs_full_sorted = obs_tilde_virtual_full[
    np.argsort(np.abs(obs_tilde_virtual_full.coeffs))[::-1]
]

# 現在の切り捨てが必要とする基底。
bases_cut, _ = get_measurement_bases(obs_full_sorted[:num_to_measure])
n_bases_cut  = len(bases_cut)
print(f"Truncation : {num_to_measure} terms  ->  {n_bases_cut} measurement bases")

# カットのすぐ先の項をたどる: 新しい基底を必要としなければ、その項は「ただ」。
free = 0
for k in range(num_to_measure + 1, len(obs_full_sorted) + 1):
    bases_k, _ = get_measurement_bases(obs_full_sorted[:k])
    if len(bases_k) == n_bases_cut:
        free += 1
    else:
        break  # 新しい基底を必要とする最初の項

if free:
    print(f"Past cut   : {free} further term(s) fit the existing {n_bases_cut} "
          f"bases -- keeping {num_to_measure + free} terms costs no extra basis.")
else:
    print("Past cut   : the next term needs a new basis -- no free terms here.")
print("(Free in *basis* cost; each extra term still adds its own shot noise.)")

In [ ]:
# 実行中のショット数を制御する
shots_per_randomization_exec = 64
num_randomizations_exec = 1024

# 実行中に実際には注入されないようにノイズをゼロにする。
# PNA が回路の層にノイズを関連付けられるようにするためだけに、
# InjectNoise アノテーションを追加した。
samplex_inputs_pna = {f"noise_scales.{ref}": 0.0 for ref in refs_to_noise_models_pna}
samplex_inputs_pna |= {"pauli_lindblad_maps": refs_to_noise_models_pna}

# 測定する基底を指定する。
# ref 名 "basis2" は、この特定の回路について `samplomatic.build` が
# 測定ボックスの ChangeBasis アノテーションに自動割り当てしたものである。
# 他の回路について ref 名を確認するには、次を実行する:
#     print(samplex_pna.inputs().make_broadcastable().describe())
# そして "basis_changes.<ref>" で始まる行を探す。

bases_broadcastable = np.expand_dims(np.array(meas_bases_canonical), axis=1)

# 自動生成される basis_changes に対応するために追加
change_basis_ann = get_annotation(meas_box.operation, ChangeBasis)
basis_ref = change_basis_ann.ref

samplex_inputs_pna |= {"basis_changes": {basis_ref: bases_broadcastable}}

In [ ]:
# samplex_inputs を QuantumProgram に渡す辞書に変換する。
samplex_arguments_pna = samplex_pna.inputs().make_broadcastable().bind(**samplex_inputs_pna)

# 指定したパラメーターで QuantumProgram をインスタンス化する。
program = QuantumProgram(shots=shots_per_randomization_exec)
program.append_samplex_item(
    circuit=template_circuit_pna,
    samplex=samplex_pna,
    samplex_arguments=samplex_arguments_pna,
    shape=(num_randomizations_exec,),
)

#### Executor ジョブを実行する

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `EXECUTOR_JOB_ID_PNA` パラメーターに貼り付け、`SUBMIT_EXECUTOR_JOB_PNA = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは30秒です（ibm_fez でテスト済み）。_ この使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
executor = Executor(backend)

EXECUTOR_JOB_ID_PNA      = None      # paste an Executor job_id here on re-run
SUBMIT_EXECUTOR_JOB_PNA  = False    # set True to submit a fresh Executor job
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_PNA is not None:
    job_exec_pna = service.job(EXECUTOR_JOB_ID_PNA)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_PNA}")
elif SUBMIT_EXECUTOR_JOB_PNA:
    job_exec_pna = executor.run(program)
    EXECUTOR_JOB_ID_PNA = job_exec_pna.job_id()      
    print(f"Submitted: {EXECUTOR_JOB_ID_PNA}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_PNA=True to submit a fresh job, "
          "or paste a saved Executor job id into EXECUTOR_JOB_ID_PNA and re-run.")

In [ ]:
# ジョブの種類と状態を確認する
if EXECUTOR_JOB_ID_PNA is not None:
    print(f"Program id  : {job_exec_pna.job_id()}")
    print(f"Status      : {job_exec_pna.status()}")

ジョブの結果を取り出します。

In [ ]:
if job_exec_pna.status() == "DONE":
    exec_results_pna = job_exec_pna.result()
else:
    print(f"Not done yet (status={job_exec_pna.status()}). Re-run cell when DONE.")

#### TREX エラー緩和の手法を追加する

1.1.1 節と 1.1.2 節では、[Twirled readout error extinction（TREX）](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) を、`Estimator` プリミティブで簡単にオンにできるエラー緩和の手法として紹介しました。TREX は、測定の前にランダムに $X$ ゲートを挿入し、結果を古典的に反転することで、測定からの読み出しエラーを緩和します。これは読み出しエラー行列を対角化し、読み出しエラーを緩和できるようにします。

ここ PNA ワークフローでは、[Qiskit add-ons utils ライブラリー](https://github.com/Qiskit/qiskit-addon-utils) の関数 `trex_factors` を使って TREX を簡単に実装できます。

In [ ]:
# TREX 係数を計算する
measurement_noise_map = noise_learner_result_pna[2].to_pauli_lindblad_map()
trex_rescale_factors = trex_factors(measurement_noise_map, bases_reverser)

### 3.1.5  結果を分析する

この節では、磁化の期待値の3つの推定値を比較します。緩和なし、PNA のみ、そして PNA + TREX です。ミラーの出力状態 $|0\rangle^{\otimes N}$ 上の期待値の理想値は $N=10$ に等しいです。

Qiskit add-on utils ライブラリーの関数 `executor_expectation_values` を使って、3つのケースすべてで期待値を計算します。

まず、PNA のみでエラー緩和された期待値から始めましょう。

In [ ]:
# PNA のみの期待値。
# `bases_reverser` は `obs_tilde_virtual_ex4` から構築されたので、各測定基底を
# その基底で対角な（PNA 後の）パウリ項に対応付ける。
#
# データの軸0の長さは len(meas_bases_canonical) であり、len(bases_reverser) より
# 1つ大きいことがある — 緩和なしのベースラインのために all-Z 基底を追加したため。
# 渡す前に、データを元の PNA 基底に切り詰める。

n_pna_bases = len(bases_reverser)

results_pna = executor_expectation_values(
    exec_results_pna[0]["meas"][:n_pna_bases],
    bases_reverser,
    meas_basis_axis=0,
    avg_axis=1,
    measurement_flips=exec_results_pna[0]["measurement_flips.meas"][:n_pna_bases],
    pauli_signs=None,  # noise_scales=0 ⇒ PEC なし ⇒ パウリ符号なし
    rescale_factors=None,
)

exp_val_pna = results_pna[0][0]
std_pna     = results_pna[0][1]
print(f"PNA only    : <O_Z> = {exp_val_pna:+.4f}  std = {std_pna:.4e}")

次に、ノイズを含む緩和なしの期待値を計算します。

In [ ]:
# ─── 緩和なしの期待値 ───
# 上で all-Z 基底を meas_bases_canonical に追加した（インデックス `all_z_idx`）ので、
# executor は同じジョブの一部としてそれを測定した。データのその部分集合を切り出し、
# 元のオブザーバブル O_Z と対応付ける。

meas_z  = exec_results_pna[0]["meas"][all_z_idx]
flips_z = exec_results_pna[0]["measurement_flips.meas"][all_z_idx]

bases_reverser_unmit = {Pauli("Z" * num_qubits_pna): [target_observable_ex4]}

res_unmit = executor_expectation_values(
    meas_z,
    bases_reverser_unmit,
    meas_basis_axis=None,   # すでに切り出し済み — 基底軸は残っていない
    avg_axis=0,              # ランダム化にわたって平均する（いまは軸0）
    measurement_flips=flips_z,
    pauli_signs=None,        # noise_scales=0 ⇒ PEC なし ⇒ パウリ符号なし
    rescale_factors=None,
)
exp_val_unmit, std_unmit = res_unmit[0][0], res_unmit[0][1]
print(f"Unmitigated : <O_Z> = {exp_val_unmit:+.4f}  std = {std_unmit:.4e}")

最後に、PNA と TREX の両方のエラー緩和の手法を使って期待値を計算します。

In [ ]:
# ─── PNA + TREX ───
# TREX は、NoiseLearnerV3 の測定層モデルが学習した読み出し（測定）ノイズを吸収する
# ために、データを基底ごとにリスケーリングする。上と同様に、データを元の PNA 基底に
# 切り詰める（緩和なしのベースラインのためだけに追加した all-Z 基底を飛ばす）。
measurement_noise_map = noise_learner_result_pna[2].to_pauli_lindblad_map()
trex_rescale_factors  = trex_factors(measurement_noise_map, bases_reverser)

n_pna_bases = len(bases_reverser)

results_pna_trex = executor_expectation_values(
    exec_results_pna[0]["meas"][:n_pna_bases],
    bases_reverser,
    meas_basis_axis=0,
    avg_axis=1,
    measurement_flips=exec_results_pna[0]["measurement_flips.meas"][:n_pna_bases],
    pauli_signs=None,  # noise_scales=0 ⇒ PEC なし ⇒ パウリ符号なし
    rescale_factors=trex_rescale_factors,
)

exp_val_pna_trex = results_pna_trex[0][0]
std_pna_trex     = results_pna_trex[0][1]
print(f"PNA + TREX  : <O_Z> = {exp_val_pna_trex:+.4f}  std = {std_pna_trex:.4e}")

結果をプロットします。

In [ ]:
# 比較: 緩和なし、PNA、PNA+TREX vs. 理想値（10量子ビット鎖では = N = 10）。
experiments = ["Unmitigated", "PNA", "PNA+TREX"]
colors      = ["tab:gray", "tab:blue", "tab:orange"]
markers     = ["o", "s", "^"]

evs  = [exp_val_unmit, exp_val_pna, exp_val_pna_trex]
errs = [std_unmit,     std_pna,     std_pna_trex]
x    = np.arange(len(experiments))

plt.figure(figsize=(6, 4))
for xi, yi, ei, label, color, marker in zip(x, evs, errs, experiments, colors, markers):
    plt.errorbar(
        xi, yi, yerr=ei,
        color=color, marker=marker, markersize=12,
        linestyle="none", capsize=5,
        label=label, zorder=3,
    )

plt.axhline(y=num_qubits_pna, color="green", linestyle="--", linewidth=2,
            label=f"Ideal = N = {num_qubits_pna}", zorder=2)

plt.xticks(x, experiments)
plt.ylabel("Expectation value", fontsize=14)
plt.title(r"10 qubit Ising chain, 2 Trotter steps, $O_Z$ obs",
          fontsize=14)
plt.legend(loc="lower right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

上のプロットでは、3つの実装で結果にはっきりとした違いが見えるはずです。緩和なしの期待値が最も悪いはずです（すなわち、理想値 10 から最も遠い）。緩和なしのオブザーバブル $O$ の代わりにノイズ緩和オブザーバブル $\tilde{O}$ の期待値を測定して PNA を実装すると、結果が改善するはずです。最後に、PNA に加えて TREX も追加して読み出しエラーも緩和すると、結果がさらに改善するのが見えるはずです。

これで propagated noise absorption（PNA）を扱った 3.1 節が終わります。3.2 節では、PEC と SLC に移ります。

## 3.2 Probabilistic Error Cancellation（PEC）と Shaded lightcones（SLC）

### 3.2.1 Probabilistic Error Cancellation（PEC）の概要
[Probabilistic error cancellation（PEC）](https://quantum.cloud.ibm.com/docs/en/guides/error-mitigation-and-suppression-techniques#probabilistic-error-cancellation-pec) は回路のレベルで作用します。第2章の学習されたパウリノイズモデルを使って逆ノイズ写像（反ノイズ）を構築し、その後、疑確率（quasi-probability）の重みからショットごとの符号を付けてその逆からサンプリングします。最後に $\gamma$ でリスケーリングした後、平均された期待値は、ノイズモデルが正確であれば理想的な回路に対して不偏です。

概念的には、PEC は PNA の回路側の類似物です。
- **PNA** は回路を固定したままオブザーバブルを書き換えます。
- **PEC** はオブザーバブルを固定したまま回路を確率的に書き換えます。

PEC は一般的です。正確なノイズモデルがあれば、オブザーバブルに特別な構造を要求せずにエラーを緩和します。この一般性の代償は、推定量の分散の増加です。ショット数は回路の深さと総ノイズとともに増え、オーバーヘッド係数 $\gamma^2$ で要約されます。定量的には、$$\gamma = \exp\!\big(2\sum_{l,\sigma}\lambda_{l,\sigma}\big),$$ ここで $\lambda_{l,\sigma}$ は回路の層 $l$ における生成子 $\sigma$ の学習されたレートです。サンプリングコストは $\gamma^2$ として増大します。

1.1.2 節で見たように、Runtime `Estimator` はワンフラグの PEC オプション（`resilience.pec_mitigation = True`）を公開しています。このスイッチは便利で、*内部的には* 層ごとのノイズモデルを使いますが、層ごとや生成子ごとのつまみをあなたに公開せず、すべての学習された生成子が打ち消されるため、サンプリングオーバーヘッド $\gamma^2$ が深さとともに指数的に膨れ上がりうります。

この節では、**Shaded lightcones**（SLC）アドオンを使って、$\gamma^2$ を実用規模の回路で扱える程度まで下げることに焦点を当てます。以下の節で、SLC が完全な PEC と比べてサンプリングオーバーヘッドをどう減らすかを見ます。完全な PEC を使う場合と SLC アドオンを使う場合とで、3つの異なるオブザーバブルについてサンプリングオーバーヘッドを比較します。

### 3.2.2 Shaded Lightcones（SLC）の概要

[Shaded lightcones（SLC）](https://qiskit.github.io/qiskit-addon-slc/) は PEC の発展形です。回路の _すべての_ ノイズを打ち消すのではなく、回路の各ノイズ生成子が対象のオブザーバブルにどれだけ影響するかを境界付けます。境界付けられた影響が小さい生成子はスケールダウンするか飛ばすことで、同じ残留バイアスのもとで PEC のサンプリングオーバーヘッド $\gamma^2$ を減らします。

その直観は因果的です。回路の末尾で測定されるオブザーバブルは、その後方ライトコーン _内の_ ノイズ、すなわち摂動が測定に伝播しうる場所の集合からしか影響を受けません。ライトコーンの外のエラーは測定結果に影響できないので、エラーキャンセルの手続きから除外できます。SLC は標準的な二値のライトコーンを超えます。各ノイズ生成子に、（コーンの内か外かの二値ではなく）オブザーバブルをどれだけ強くずらしうるかの境界を割り当てます。基礎となる手法については [Lightcone Shading for Classically Accelerated Quantum Error Mitigation](https://arxiv.org/abs/2409.04401) を参照してください。

SLC のワークフローは次のとおりです。

1. **前方境界（forward bounds）** を計算する: これは、各ノイズ生成子が存在した場合に、最終オブザーバブルに影響するよう前方にどう伝播するかを定めます。

2. **後方境界（backward bounds）** を計算する: これは、後方に発展させたオブザーバブルが、各ノイズ位置からどう寄与を拾うかを定めます。

3. **マージ**: 前方境界と後方境界を学習されたノイズモデルとマージして、エラー生成子ごとの関連度スコアを得ます。

4. それらのスコアを関数 `compute_local_scales` を通して使い、どのエラーを緩和する価値があるかを選び、制御されたバイアスをより小さい $\gamma^2$ と引き換えにします。

#### SLC アドオンのインポート:

In [ ]:
# すべての SLC アドオンのインポートは、ノートブック先頭の「セットアップとインポート」で読み込み済み

# 必要なインポートがすべてそろっているかの簡単な健全性チェック:
required = [
    "compute_backward_bounds", "compute_forward_bounds", "compute_local_scales",
    "merge_bounds", "generate_noise_model_paulis", "map_modifier_ref_to_ref",
    "draw_shaded_lightcone",
]
missing = [name for name in required if name not in globals()]
if missing:
    raise ImportError(
        f"Missing SLC names: {missing}. Re-run the master imports cell at the "
        f"top of the notebook."
    )
print("SLC imports OK.")

### 3.2.3 セットアップと局所オブザーバブル

SLC の効果を見るために、ライトコーンを観察できるほど十分に深い例の回路を見たいと思います。この目的のために、イジングミラー回路の例を引き続き使い、トロッターステップ数を10に増やします。ステップが多いほど回路が深くなり、したがってライトコーンも深くなります。

- `num_qubits_pna` $= 10$
- トロッターステップ数を10に増やす（`num_trotter_steps_slc` $= 10$）
- `rx_angle` = $\pi / 8$ を保つ

_注意:_ SLC は、3.1 節で PNA に使ったものとは異なるノイズ注入戦略を必要とします。PNA の `uniform_modification` は1つの大域的な `noise_scales` 枠を公開しますが、SLC は各パウリ生成子をオブザーバブルのライトコーンに沿ってスケーリングするために *生成子ごとの* 制御を必要とします。SLC の `inject_noise_strategy="individual_modification"` は、`InjectNoise` アノテーションごとに別々の `noise_scales.<ref>` 枠を公開します。

In [ ]:
# individual modification 付きの SLC 用の新しいボックス化パスマネージャー
slc_boxing_pm = generate_boxing_pass_manager(
    enable_gates=True,
    enable_measures=True,
    measure_annotations="all",
    twirling_strategy="active",
    inject_noise_targets="gates",
    inject_noise_strategy="individual_modification",
)

### 対象のオブザーバブル

この節で考えるオブザーバブルは、局所オブザーバブル $O = Z_4$、すなわち10量子ビットのイジング鎖の量子ビット4上の $Z$ です。このオブザーバブルは、10量子ビットのうち1つにしか作用しないので _局所的_ です。

In [ ]:
# 局所オブザーバブル: 中央の量子ビット上の Z、それ以外は恒等。
target_obs_local: list[tuple[str, list[int], float]] = [("Z", [4], 1.0)]

# オブザーバブルを構築する。
observable_local = SparsePauliOp.from_sparse_list(target_obs_local, num_qubits=num_qubits_pna)
print(observable_local)
 
# 測定基底を構築する。 
bases_virt, reverser_virt = get_measurement_bases(observable_local)

#### なぜ SLC は局所オブザーバブルに適しているのか？

SLC の手法は、量子回路の因果構造を活用するため、局所オブザーバブルに特に適しています。局所オブザーバブルは量子ビットの小さな部分集合にしか作用しないので、SLC は関連するライトコーン、すなわちオブザーバブルの期待値に因果的に影響するゲートの部分集合を効率的に特定して集中できます。このライトコーンの外のエラーは安全に無視でき、計算のオーバーヘッドを劇的に減らします。この的を絞ったアプローチによって、SLC は、すべてのエラーを追跡しなければならない手法と比べて、局所オブザーバブルに対してより効率的になります。

対照的に、10量子ビットすべてに作用するオブザーバブルを考えると、対応するライトコーンは回路のより広い部分にわたります。その結果、SLC は回路のより多くのゲートのエラーを追跡する必要があり、PEC 単独と比べてサンプリングオーバーヘッドの削減があまり得られないかもしれません。

演習5では、異なる局所性の度合いを持つ3つの異なるオブザーバブルを考えます。オブザーバブルの局所性が SLC のサンプリングオーバーヘッドにどう影響するかを、自分で目の当たりにします。

局所オブザーバブルを定義したので、このラボを通してこれまで何度も使ってきた手順に従って問題をセットアップする必要があります。

1. イジング回路を作る（今回は10トロッターステップ）
2. 回路をミラーし、末尾ですべての量子ビットを測定する
3. ミラー化された回路をバックエンドの ISA に従うようにトランスパイルする
4. オブザーバブルを ISA レイアウトにマップする
5. 回路をボックス化し、回路のユニークな層を見つける

これら5つのステップすべてを下のセルで実装します。

In [ ]:
num_trotter_steps_slc = 10 # ライトコーンを見るために10に増やす

# ミラー化されたイジング回路を構築する。
ising_slc = construct_ising_circuit(num_qubits_pna, num_trotter_steps_slc, rx_angle)
mirror_slc = ising_slc.compose(ising_slc.inverse())
mirror_slc.measure_all()

# バックエンドの ISA にトランスパイルし、レイアウトを取得する。
mirror_isa_slc = isa_pm.run(mirror_slc)

# オブザーバブルを ISA レイアウトにマップする 
observable_local_isa = observable_local.apply_layout(mirror_isa_slc.layout, num_qubits=mirror_isa_slc.num_qubits)
layout_slc = mirror_isa_slc.layout.final_index_layout()
wire_order_slc = layout_slc + [q for q in range(mirror_isa_slc.num_qubits) if q not in layout_slc]

# SLC 用にボックス化する（後でエラー項ごとにスケーリングできるように `individual_modification` を使う）。
boxed_circuit_slc = slc_boxing_pm.run(mirror_isa_slc)
unique_layer_slc = find_unique_box_instructions(
    boxed_circuit_slc, normalize_annotations=None, undress_boxes=True
)

### 3.2.4 学習されることになるノイズモデルのパウリを予測する

[Qiskit SLC アドオン](https://qiskit.github.io/qiskit-addon-slc/) の関数 `generate_noise_model_paulis` を使って、回路のパウリレベルのノイズ記述を構築します。

- このステップは、いかなるノイズ学習の知識も _なしに_ 実行されます。
- この関数は回路の各ボックス化された層をスキャンします。
- 関連するすべての __重み1__ と __重み2__ のパウリノイズ項を生成します。
- パウリ項は回路の量子ビット接続性によって制限されます（アクティブな量子ビットとエッジ上でサポートされる項だけが保たれます）
- 結果として得られるパウリの集合は、ノイズの強さとは独立に、可能なノイズのサポートを定めます。
- これらのパウリ項は、shaded lightcones を定める __前方ノイズ境界と後方ノイズ境界__ を計算するために使われます。
- 後に、ノイズ学習によって、この固定されたパウリ基底に __層ごとのパウリ・リンドブラッド係数__ を割り当てます。

順序: SLC はまず、回路構造とオブザーバブルの局所性から、ノイズがどこで問題になりうるかを決めます。ノイズ学習は後に、それがどれだけ問題になるかを決めます。

In [ ]:
noise_model_paulis = generate_noise_model_paulis(unique_layer_slc, backend.coupling_map, boxed_circuit_slc)
noise_model_rates = {ref: None for ref in noise_model_paulis}

### 3.2.5 前方境界と後方境界を計算する

前方境界と後方境界を計算して、回路のどの部分がオブザーバブルに影響しうるかを決めます。設定を調整するとライトコーンの形と色が変わり、異なる回路領域が最終測定にどう影響するかが分かります。この分析はノイズ学習の前にシミュレーションで実行されるので、QPU 時間は使いません。

境界は `compute_forward_bounds` と `compute_backward_bounds` 関数を使って計算されます。これらの関数は、ボックス化された回路とノイズモデルのパウリ集合を受け取ります。`compute_forward_bounds` 関数は、ISA レイアウトにマップされたオブザーバブルも受け取ります。

SLC 境界計算のために追加のパラメーターをいくつか定義します。
- `slc_atol` - 発展中に小さいパウリ項を切り捨てるための絶対許容誤差。この閾値を下回る係数を持つ項は、計算の複雑さを抑えるために破棄されます。
- `slc_eigval_max_qubits` - 固有値の計算をなお試みる交換子の最大量子ビット数。この値を超えると、境界はより単純で緩い三角不等式で近似されます。
- `slc_evolution_max_terms` - 時間発展中に保持するパウリ項の最大数。発展した演算子のサイズを制限し、複雑さの指数的な増大を防ぎます。
- `slc_num_processes` - 使う並列プロセスの数。並列計算を可能にして前方/後方境界の計算を高速化します。
- `slc_timeout` - 各境界計算に許される最大時間（秒）。個々の計算が無期限に走るのを防ぎます。

これらのパラメーター選択の詳細については、[`compute_forward_bounds`](https://qiskit.github.io/qiskit-addon-slc/apidocs/qiskit_addon_slc.bounds.html#qiskit_addon_slc.bounds.compute_forward_bounds) と [`compute_backward_bounds`](https://qiskit.github.io/qiskit-addon-slc/apidocs/qiskit_addon_slc.bounds.html#qiskit_addon_slc.bounds.compute_backward_bounds) 関数のドキュメントを参照してください。

In [ ]:
slc_atol = 1e-8
slc_eigval_max_qubits = 18
slc_evolution_max_terms = 1000
slc_num_processes = 8
slc_timeout = 300

_注意_: 以下のセルは、あなたのコンピューターでローカルに実行した場合、約45秒かかります。`qBraid` のようなクラウドサービスを使っている場合、最大5分かかることがあります。必要なら、上の `slc_timeout` をより長い時間に調整してください。

In [ ]:
forward_bounds = compute_forward_bounds(
    boxed_circuit_slc,
    noise_model_paulis,
    observable_local_isa,
    evolution_max_terms=slc_evolution_max_terms,
    eigval_max_qubits=slc_eigval_max_qubits,
    atol=slc_atol,
    num_processes=slc_num_processes,
    timeout=slc_timeout,
)

backward_bounds = compute_backward_bounds(
    boxed_circuit_slc,
    noise_model_paulis,
    evolution_max_terms=slc_evolution_max_terms,
    num_processes=slc_num_processes,
    timeout=slc_timeout,
)

これで前方ライトコーンと後方ライトコーンをプロットできます。

In [ ]:
print("forward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_slc,
            forward_bounds,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )

print("backward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_slc,
            backward_bounds,
            noise_model_paulis,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order_slc,
            measure_arrows=False,
        )
    )

黄色い領域はノイズへの感度が高いことを示し（ここでのエラーはオブザーバブルへの影響がより強い）、暗い領域は感度が低いことを示します（ここでのエラーはあまり問題になりません）。重要なことに、これらの図は実際のノイズがどこにあるかを示すものではありません。まだ学習していないからです。そうではなく、与えられた場所のノイズが、もし存在した場合に、対象のオブザーバブルにどれだけ強く影響するかを示しています。

次の節では、この例を15量子ビットにスケールアップします。ハードウェア上でノイズを学習し、それを shaded lightcones と組み合わせて、低いサンプリングオーバーヘッドでエラーを緩和します。

### 演習5 — 15量子ビットについて3つのオブザーバブルの局所性を調べる

10量子ビット、10トロッターステップ、オブザーバブル $Z_4$ のイジングハミルトニアンについて、shaded lightcones（前方境界と後方境界）を見てきました。この演習では、例を15量子ビットにスケールしますが、トロッターステップ数は3に減らします。

SLC の PEC に対する優位性は、削減されたサンプリングオーバーヘッドから来ます。これはオブザーバブルが _局所的_ なときに特に効果的です。サンプリングオーバーヘッドの削減は、オブザーバブルの局所性と複雑さに依存します。この演習では、異なる局所性を持つ3つのオブザーバブルを構築し、対応する前方境界と後方境界を計算します。

3トロッターステップの15量子ビットのミラー化イジング回路をセットアップし、3つのオブザーバブルを構築します。
- `obs_Z7` - 量子ビット7上の Z
- `obs_X3_Z11` - 量子ビット3上の X、量子ビット11上の Z
- `obs_7_Zs` - 中央の7量子ビット上の Z

これらのオブザーバブルそれぞれについて、前方境界と後方境界を計算する必要があります。

<div class="alert alert-block alert-success">

<b>演習5。</b> 15量子ビットについて3つのオブザーバブルの局所性を調べる

- **3トロッターステップの15量子ビットのイジング鎖のミラー化回路を作る。**
    - 関数 `construct_ising_circuit` を使う
- **必要な3つのオブザーバブルを `SparsePauliOp` オブジェクトとして作る**:
    - `obs_Z7` - 量子ビット7上の Z
    - `obs_X3_Z11` - 量子ビット3上の X、量子ビット11上の Z
    - `obs_7_Zs` - イジング鎖の中央の7量子ビット上の Z
    - 各オブザーバブルは、係数 1 の単一項のオブザーバブルであるべきです。
- **shaded light cone 境界を計算する:**
    - 3つのオブザーバブルそれぞれについて:
        - オブザーバブルを物理（ISA）量子ビットレイアウトにマップする。
        - `compute_forward_bounds` を使って前方境界を計算する
        - 前方境界を `forward_bounds_list` に追加する
    - 後方境界はオブザーバブルに依存しないので、一度だけ計算すればよいです:
        - `compute_backward_bounds` を使って後方境界を計算する


下のセルを埋めてください。

</div>

_注意:_ 前方境界と後方境界の計算には約30秒かかるので、辛抱してください！

In [ ]:
# ========== 演習 ==========

# ============================================================
# TODO: 15量子ビットのイジング回路を作る
# 下の必要なコードを埋めて
# ============================================================

# 量子ビット数とトロッターステップ数を定義する。
num_qubits_ex5        = 15
num_trotter_steps_ex5 = 3
rx_angle         = np.pi / 8

# ミラー化されたイジング回路を構築する。
circuit_ising_ex5 = ...
mirrored_circuit_ex5 = ...
mirrored_circuit_ex5.measure_all()

# バックエンドの ISA にトランスパイルし、レイアウトを取得する。
isa_circuit_ex5 = ...
final_layout_ex5 = isa_circuit_ex5.layout.final_index_layout()

# SLC 用にボックス化する（後でエラー項ごとにスケーリングできるように `individual_modification` を使う）。
boxed_circuit_ex5 = slc_boxing_pm.run(isa_circuit_ex5)
unique_layers_ex5 = find_unique_box_instructions(
    boxed_circuit_ex5, normalize_annotations=None, undress_boxes=True
)

# ノイズモデルのパウリ項を生成する。
noise_model_paulis_ex5 = generate_noise_model_paulis(unique_layers_ex5, backend.coupling_map, boxed_circuit_ex5)
noise_model_rates_ex5 = {ref: None for ref in noise_model_paulis_ex5}

# ============================================================
# 必要なオブザーバブルを SparsePauliOp として構築する
# ============================================================

# 量子ビット7上の Z  →  1項: ("Z", [7], 1.0)
obs_Z7 = ...

# 量子ビット3上の X、量子ビット11上の Z  →  1項: ("XZ", [3, 11], 1.0)
obs_X3_Z11 = ...

# 中央の7量子ビットそれぞれの上の Z（量子ビット4..10 上の重み7のパウリ積1つ）
obs_7_Zs = ...

obs_list = [obs_Z7, obs_X3_Z11, obs_7_Zs]

# ============================================================
# 各 obs について前方境界を計算する（obs_list をループする）。
# ============================================================

slc_atol = 1e-8
slc_eigval_max_qubits = 18
slc_evolution_max_terms = 1000
slc_num_processes = 8
slc_timeout = 600

forward_bounds_list = []

for obs in obs_list:
    # 3つのオブザーバブルすべての前方境界を計算するコードをここに書く

    # 前方境界をリストに追加する
    forward_bounds_list.append(forward_bounds)

# ========================================================================================
# 後方境界を計算する — 後方境界は対象のオブザーバブルに依存しないので、
# 一度だけ計算すればよい
# ========================================================================================

backward_bounds = ...

In [ ]:
# 解答を採点する
grade_lab3_ex5(circuit_ising_ex5, 
    mirrored_circuit_ex5, 
    boxed_circuit_ex5, 
    obs_list, 
    forward_bounds_list, 
    backward_bounds
)

In [ ]:
# すべてのラボにわたる進捗を確認する
check_progress()

下のプロットは、3つのオブザーバブルについて前方境界と後方境界を示しています。

In [ ]:
print(f"forward bounds for observable obs_Z7 :")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            forward_bounds_list[0],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )


In [ ]:
print("forward bounds for observable obs_X3_Z11 :")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            forward_bounds_list[1],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )


In [ ]:
print("forward bounds for observable obs_7_Zs :")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            forward_bounds_list[2],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )


In [ ]:
print("backward bounds:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            backward_bounds,
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            wire_order=wire_order_slc,
            measure_arrows=False,
        )
    )

### 3.2.6 マージされた境界

これで前方境界と後方境界をマージして、サンプリングオーバーヘッドを最小化する shaded lightcone を定める準備ができました。

#### なぜ境界をマージするのか？

関数 `merge_bounds` は、後方境界が前方境界に切り替わる回路内の遷移点を選びます。遷移は、推定される総バイアスを最小化するように選ばれ、きつい（tight）ライトコーンを生み出します。

これにはハードウェアから学習されたノイズレートが必要です。ノイズモデルは、回路のどの部分がエラーを支配するかをマージのアルゴリズムに教えます。

#### より大きい回路でノイズを学習し直す

演習5では15量子ビットにスケールアップしたので、回路が変わり、そのユニークな層は、以前の10量子ビットの PNA セットアップから学習したノイズモデルとは異なる物理量子ビット上にあります。したがって、境界をマージできるようにする前に、15量子ビットの層でノイズを学習し直す必要があります。PNA 用に `pna_learner` で定義したノイズ学習のパラメーターを再利用します。

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `NOISE_LEARN_JOB_ID_SLC` パラメーターに貼り付け、`SUBMIT_NOISE_JOB_SLC = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは13秒です（ibm_pittsburgh でテスト済み）。_ 上の使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
NOISE_LEARN_JOB_ID_SLC = None   # paste job_id here on re-run
SUBMIT_NOISE_JOB_SLC   = False  # set True to submit a fresh learning job
pna_learner.options.environment.job_tags = ["qgss26"]

if NOISE_LEARN_JOB_ID_SLC is not None:
    learner_job_slc = service.job(NOISE_LEARN_JOB_ID_SLC)
    print(f"Re-using saved job: {NOISE_LEARN_JOB_ID_SLC}")
elif SUBMIT_NOISE_JOB_SLC:
    learner_job_slc = pna_learner.run(unique_layers_ex5)
    NOISE_LEARN_JOB_ID_SLC = learner_job_slc.job_id()
    print(f"Submitted: {NOISE_LEARN_JOB_ID_SLC}")
else:
    print("Set SUBMIT_NOISE_JOB_SLC=True to submit a fresh job, "
          "or paste a saved job id into NOISE_LEARN_JOB_ID_SLC and re-run.")


In [ ]:
learner_job_slc = service.job(NOISE_LEARN_JOB_ID_SLC)
print(f"{NOISE_LEARN_JOB_ID_SLC}  (status: {learner_job_slc.status()})")

In [ ]:
if learner_job_slc.status() == "DONE":
    noise_learner_result_slc = learner_job_slc.result()
else:
    print(f"Not done yet (status={learner_job_slc.status()}). Re-run cell when DONE.")

In [ ]:
if 'noise_learner_result_slc' in dir() and noise_learner_result_slc is not None:
    refs_to_noise_models_slc = noise_learner_result_slc.to_dict(unique_layers_ex5, require_refs=False)
    print(f"refs_to_noise_models_slc has {len(refs_to_noise_models_slc)} entries")
else:
    print("Run the noise-learning cell above and wait for it to finish first.")


#### マージされた境界を計算する

各オブザーバブルについて、学習されたノイズモデル `refs_to_noise_models_slc` を使ってマージされた境界を計算します。

_注意:_ 次の警告メッセージが表示されることがあります。"_Optimal spacetime partitioning not implemented!Just partitioning list of noisy boxes._" これは想定内でエラーではありません。

In [ ]:
# 学習されたノイズモデル付きのマージされた境界 — 実際のパウリ・リンドブラッドレートを
# 使うので、関連するライトコーンが通常かなり縮む。
merged_bounds_with_noise = []
for i in range(3):
    merged_bounds_with_noise.append(merge_bounds(
        boxed_circuit_ex5,
        forward_bounds_list[i],
        backward_bounds,
        refs_to_noise_models_slc,
    ))

In [ ]:
print("Merged bounds with noise model for observable obs_Z7:")
for p in "XYZ":
    display(
        draw_shaded_lightcone(
            boxed_circuit_ex5,
            merged_bounds_with_noise[0],
            noise_model_paulis_ex5,
            pauli_filter=p,
            scale=0.15,
            fold=-1,
            idle_wires=False,
            measure_arrows=False,
        )
    )



### 3.2.7 サンプリングオーバーヘッドを比較する

この節では、演習5で構築した3つのオブザーバブルについて、SLC を実装する場合と PEC 単独の場合のサンプリングオーバーヘッドを計算します。

3.2.1 節で PEC のサンプリングオーバーヘッドを $\gamma^2$ として定義しました。ここで $$\gamma = \exp\!\big(2\sum_{l,\sigma}\lambda_{l,\sigma}\big),$$ で、$\lambda_{l,\sigma}$ は回路の層 $l$ における生成子 $\sigma$ の学習されたレートです。

下のセルでは、上の式からと、ヘルパー関数 `gamma_from_noisy_boxes` を使っての両方で量 $\gamma$ を計算し、それらが一致することを確認します。

In [ ]:
id_map = map_modifier_ref_to_ref(boxed_circuit_ex5)

# パウリ・リンドブラッドの式からの手計算の γ: γ = exp(2 · Σ λ)。
summed_rates = 0.0
for box_id, noise_id in id_map.items():
    learned_plm_ex5 = refs_to_noise_models_slc[noise_id]
    summed_rates += np.sum(learned_plm_ex5.rates)
total_gamma = np.exp(2 * summed_rates)

# ヘルパーと突き合わせて確認する。これはエッジケース
# （フィットされた負のレート、誤対応の ref など）を防ぐ。
gamma_full_helper = gamma_from_noisy_boxes(refs_to_noise_models_slc, id_map)
assert np.isclose(total_gamma, gamma_full_helper), (
    f"Manual γ ({total_gamma:.6e}) disagrees with gamma_from_noisy_boxes ({gamma_full_helper:.6e})"
)

print(f"Full PEC γ = {total_gamma:.6f},  sampling cost γ² = {total_gamma**2:.6f}")
print(f"(helper agrees: γ = {gamma_full_helper:.6f})")

#### 局所スケールを計算する

関数 `compute_local_scales` は、回路の可能な各ノイズ生成子を、そのエラーが最終測定のバイアスにどれだけ寄与しうるかでランク付けします。また、そのエラーを訂正する緩和コストも考慮します。定義された `bias_tolerance` の予算内で、エラー緩和がバイアスをできるだけ減らすようなノイズ生成子の部分集合を選びます。この関数は次の3つを返します。

- `local_scales`: どのパウリを積極的に緩和し、どれを放っておくかを示す生成子ごとのスケール係数
- `sampling_cost`: 予測される $\gamma^2$ オーバーヘッド
- `residual_bias_bound`: 一部の生成子を緩和しないまま残すことで保たれるバイアス

In [ ]:
# 3つのオブザーバブル（obs_Z7, obs_X3_Z11, obs_7_Zs）についてのバイアス・コストの
# トレードオフのスイープ。bias_tolerance を 0.001 から 0.101 まで 0.01 刻みで（0.0 は
# なし — 注を参照）。
biases       = [[], [], []]
costs        = [[], [], []]
local_scales = [[], [], []]

bias_tolerance_values = np.arange(0.001, 0.102, 0.01).tolist()

for i in range(3):
    for bias in bias_tolerance_values:
        try:
            local_scale_, cost_, bias_ = compute_local_scales(
                boxed_circuit_ex5,                # ノイズボックス付きの回路
                merged_bounds_with_noise[i],      # オブザーバブル i のマージされた境界
                refs_to_noise_models_slc,         # 学習されたノイズモデルのレート
                sampling_cost_budget=np.inf,      # コスト予算なし — 最適なトレードオフを見つける
                bias_tolerance=bias,              # 許容される最大バイアス
            )
            biases[i].append(bias_)
            costs[i].append(cost_)
            local_scales[i].append(local_scale_)
        except (IndexError, ValueError):
            # compute_local_scales はすべての境界/バイアスの組み合わせで収束するとは
            # 限らない（境界の構造に依存する）。曲線の残りが描画されるように、
            # 静かにスキップする。
            continue

In [ ]:
xticks = np.arange(0, 11)

fig, ax = plt.subplots(figsize=(8, 5))

# 完全な PEC の基準（SLC の切り捨てなし）。
ax.axhline(y=total_gamma ** 2, color="tab:orange", linestyle="--", linewidth=2, label="Full PEC")

# 3つのオブザーバブルすべてのバイアス・コストのトレードオフ曲線。
observable_names = ["Z on q7", "X on q3, Z on q11", "Z on 7 central qubits"]
colors           = ["tab:blue", "tab:green", "tab:red"]

for i in range(3):
    if not biases[i]:
        continue  # このオブザーバブルではスイープで点が得られなかった
    ax.plot(
        100 * np.array(biases[i]),
        np.array(costs[i]),
        "o-",
        c=colors[i],
        label=f"SLC ({observable_names[i]})",
    )

ax.set_yscale("log")
ax.set_xticks(xticks, [f"{x:.1f}" for x in xticks])
ax.set_xlabel("Remaining bias [%]", fontsize=12)
ax.set_ylabel(r"Sampling overhead, $\gamma^2$", fontsize=12)
ax.grid(True, which="both", alpha=0.3)
ax.legend(loc="upper right")

fig.suptitle("PEC sampling-overhead reduction enabled by SLC", fontsize=13)
plt.tight_layout()
plt.show()

プロットは、3つのオブザーバブルそれぞれについて、サンプリングオーバーヘッド $\gamma^2$ 対 残りのバイアスを示し、完全な PEC の値を水平の基準として示しています。いくつかの要点:

- 量子ビット7上の局所単一量子ビットオブザーバブル（青）が最小のオーバーヘッドを持つ
- 2体オブザーバブル、すなわち量子ビット3上の X と量子ビット11上の Z（緑）は、その前方ライトコーンと後方ライトコーンがより多くの層を覆うので、より高い位置にある
- 量子ビット4〜10上の Z である7量子ビットオブザーバブル（赤）は、最も広いライトコーンを持つ。
- スキャンしたバイアス許容範囲全体にわたって、3つのオーバーヘッドすべてが完全な PEC を下回ったままです。

次の節では、最も局所的なオブザーバブル `obs_Z7` に焦点を当て、完全な PEC と SLC を比較するために QPU 上で回路を実行します。

### 3.2.8 QPU でエラー緩和済みジョブを実行する

1つのオブザーバブル、局所単一量子ビットの `obs_Z7` について QPU ジョブを実行します。

In [ ]:
# 動作点: bias_tolerance = 0.02（サンプリングコストの削減と残留バイアス境界の間の
# 推奨されるトレードオフ）。オブザーバブル: obs_Z7。
chosen_bias_thres = 0.02

local_scales_chosen, sampling_cost_chosen, residual_bias_chosen = compute_local_scales(
    boxed_circuit_ex5,
    merged_bounds_with_noise[0],   # obs_Z7 → インデックス 0
    refs_to_noise_models_slc,
    sampling_cost_budget=np.inf,
    bias_tolerance=chosen_bias_thres,
)
print(f"Operating point: bias_tolerance = {chosen_bias_thres}")
print(f"  sampling cost (γ²)  = {sampling_cost_chosen:.3f}  "
      f"(vs full PEC γ² = {total_gamma**2:.3f})")
print(f"  residual bias bound = {100*residual_bias_chosen:.1f}%")

ボックス化された回路の `ChangeBasis` アノテーションから基底参照を取り出し、`samplex_inputs_slc` を通して測定基底を束縛します。

- 基底のエンコード: $0=I$、$1=Z$、$2=X$、$3=Y$。

（`get_annotation` と `ChangeBasis` はノートブックの先頭でインポートされています）

In [ ]:
# boxed_circuit_ex5（15量子ビット）用のテンプレート回路 + samplex をビルドする。
template_circuit_slc, samplex_slc = samplomatic.build(boxed_circuit_ex5)

# 測定ボックスを見つけ、その ChangeBasis の ref を取り出す。
meas_boxes_slc = [
    inst for inst in boxed_circuit_ex5.data
    if inst.operation.name == "box"
    and get_annotation(inst.operation, ChangeBasis) is not None
]


basis_ref_slc = get_annotation(meas_boxes_slc[0].operation, ChangeBasis).ref


# boxed_circuit_ex5 の 正準 → 物理 → 仮想 の対応付け。
canonical_qubits_ex5 = [
    i for i, q in enumerate(boxed_circuit_ex5.qubits)
    if q in meas_boxes_slc[0].qubits
]
p_2_v_ex5 = {p: v for v, p in enumerate(final_layout_ex5)}
c_2_v_ex5 = {c: p_2_v_ex5[p] for c, p in enumerate(canonical_qubits_ex5)}

# obs_Z7 の基底（単一のパウリ項 → 単一の基底）を正準順で。
bases_virt_obs1, reverser_virt_obs1 = get_measurement_bases(obs_Z7)
meas_bases_slc_canonical = [
    np.array([base[c_2_v_ex5[c]] for c in range(15)], dtype=np.uint8)
    for base in bases_virt_obs1
]

In [ ]:
# 同じ Executor ジョブ内の2つの実験のために samplex_inputs を構築する:
#   - "unmitigated" : noise_scales = 0  （反ノイズを注入しない）
#   - "PEC+SLC"     : noise_scales = -1 + local_scales （反ノイズ、枝刈り済み）

# 緩和なし: すべてのインスタンスごとの ref で noise_scales = 0。
samplex_inputs_unmit  = {f"noise_scales.{ref}": 0.0 for ref in local_scales_chosen}
samplex_inputs_unmit |= {"basis_changes": {basis_ref_slc: meas_bases_slc_canonical[0]}}
samplex_args_unmit = samplex_slc.inputs().make_broadcastable().bind(**samplex_inputs_unmit)

# PEC+SLC: SLC の枝刈りのために noise_scales = -1 + local_scales。
samplex_inputs_slc  = {f"noise_scales.{ref}": -1.0 for ref in local_scales_chosen}
samplex_inputs_slc |= {"basis_changes": {basis_ref_slc: meas_bases_slc_canonical[0]}}
samplex_inputs_slc |= {"local_scales": local_scales_chosen}
samplex_args_slc = samplex_slc.inputs().make_broadcastable().bind(**samplex_inputs_slc)

In [ ]:
# SLC executor ジョブのパラメーター — 最小の QPU できつい統計になるよう調整。
# 各 samplex_item はこれだけのショットを得る。item は2つ（unmit + PEC+SLC）。
shots_per_randomization_slc = 64
num_randomizations_slc      = 256

# QuantumProgram を構築する。noise_maps はプログラムレベルで — SLC NoiseLearner の
# 結果からのユニークな層の ref でキー付けされる。
program_slc = QuantumProgram(
    shots=shots_per_randomization_slc,
    noise_maps=refs_to_noise_models_slc,
)

# item 0: 緩和なしのベースライン（noise_scales = 0）
program_slc.append_samplex_item(
    circuit=template_circuit_slc,
    samplex=samplex_slc,
    samplex_arguments=samplex_args_unmit,
    shape=(num_randomizations_slc,),
)
# item 1: PEC+SLC（noise_scales = -1、local_scales 付き）
program_slc.append_samplex_item(
    circuit=template_circuit_slc,
    samplex=samplex_slc,
    samplex_arguments=samplex_args_slc,
    shape=(num_randomizations_slc,),
)

print(f"Program built: 2 samplex_items (unmit + PEC+SLC), "
      f"{num_randomizations_slc} randomizations × {shots_per_randomization_slc} shots each")

_注意:_ 以下のセルは量子ハードウェア上でジョブを実行します。進む前に、その準備ができていることを確認してください。最初の実行後、ジョブ ID を `EXECUTOR_JOB_ID_SLC` パラメーターに貼り付け、`SUBMIT_EXECUTOR_JOB_SLC = False` に設定することをおすすめします。これによって、うっかりジョブを再投入するのを防ぎ、カーネル再起動後もノートブックを再実行可能に保てます。

_QPU 実行時間の見積もりは12秒です（ibm_fez でテスト済み）。_ 上の使用量の見積もりはバックエンドの実行時間のみを反映しています。キュー待ち時間、キャリブレーション、ランタイムセッションの遅延はこれより長くなる場合があります。

In [ ]:
#  qpu time = 12s, tested on ibm_fez

executor = Executor(backend)

EXECUTOR_JOB_ID_SLC      = None    # 最初の投入後にここに job_id を貼り付ける
SUBMIT_EXECUTOR_JOB_SLC  = False     # 新しいジョブを投入するには True に設定する
executor.options.environment.job_tags = ["qgss26"]

if EXECUTOR_JOB_ID_SLC is not None:
    job_exec_slc = service.job(EXECUTOR_JOB_ID_SLC)
    print(f"Re-using saved job: {EXECUTOR_JOB_ID_SLC}")
elif SUBMIT_EXECUTOR_JOB_SLC:
    job_exec_slc = executor.run(program_slc)
    EXECUTOR_JOB_ID_SLC = job_exec_slc.job_id()
    print(f"Submitted: {EXECUTOR_JOB_ID_SLC}")
else:
    print("Set SUBMIT_EXECUTOR_JOB_SLC=True to submit a fresh job, "
          "or paste a saved job id into EXECUTOR_JOB_ID_SLC and re-run.")

In [ ]:
if EXECUTOR_JOB_ID_SLC is not None:
    if job_exec_slc.status() == "DONE":
        exec_results_slc = job_exec_slc.result()
        print(f"Got {len(exec_results_slc)} samplex_item results "
              f"(item 0 = unmitigated, item 1 = PEC+SLC).")
    else:
        print(f"Not done yet: {job_exec_slc.status()}")

### 3.2.9 結果を分析する

In [ ]:
# 同じ Executor ジョブから2つの結果を取り出す:
#   results[0] → 緩和なし（noise_scales=0、gamma_factor なし）
#   results[1] → PEC+SLC （noise_scales=-1、local_scales 適用済み）

# PEC+SLC 正規化のための γ
gamma_slc = gamma_from_noisy_boxes(refs_to_noise_models_slc, id_map, local_scales_chosen)

# item 0 — 緩和なし
res_unmit_slc = executor_expectation_values(
    exec_results_slc[0]["meas"],
    reverser_virt_obs1,
    meas_basis_axis=None, avg_axis=0,
    measurement_flips=exec_results_slc[0]["measurement_flips.meas"],
    pauli_signs=exec_results_slc[0].get("pauli_signs", None),
    rescale_factors=None,
    gamma_factor=None,
)
exp_val_unmit_slc = res_unmit_slc[0][0]
std_unmit_slc     = res_unmit_slc[0][1]

# item 1 — PEC+SLC
res_slc = executor_expectation_values(
    exec_results_slc[1]["meas"],
    reverser_virt_obs1,
    meas_basis_axis=None, avg_axis=0,
    measurement_flips=exec_results_slc[1]["measurement_flips.meas"],
    pauli_signs=exec_results_slc[1].get("pauli_signs", None),
    rescale_factors=None,
    gamma_factor=gamma_slc,
)
exp_val_slc = res_slc[0][0]
std_slc     = res_slc[0][1]

ideal_obs_Z7 = 1.0   # |0⟩ 初期状態のトロッターミラー上の Z@7 → +1

print(f"{'Method':<14} {'<O_Z7>':>10} {'std':>12} {'γ²':>10}")
print("─" * 50)
print(f"{'Unmitigated':<14} {exp_val_unmit_slc:>+10.4f} {std_unmit_slc:>12.4e} {'1.000':>10}")
print(f"{'PEC+SLC':<14} {exp_val_slc:>+10.4f} {std_slc:>12.4e} {gamma_slc**2:>10.3f}")
print(f"{'Ideal':<14} {ideal_obs_Z7:>+10.4f}")
print()
print(f"Sampling-overhead reduction: {total_gamma**2 / gamma_slc**2:.2f}× vs full PEC  "
      f"(γ² {total_gamma**2:.2f} → {gamma_slc**2:.2f})")
print(f"Mitigation: {(exp_val_slc - exp_val_unmit_slc) / (ideal_obs_Z7 - exp_val_unmit_slc) * 100:.1f}% "
      f"of the unmitigated error closed")

# 比較プロット
fig, ax = plt.subplots(figsize=(7, 4.5))
methods = ["Unmitigated", "PEC+SLC"]
values  = [exp_val_unmit_slc, exp_val_slc]
errs    = [std_unmit_slc,     std_slc]
colors  = ["tab:gray", "tab:purple"]

x = np.arange(len(methods))
ax.errorbar(x, values, yerr=errs, fmt="o", markersize=14, capsize=6,
            color="black", ecolor="gray", linewidth=2, zorder=3)
for xi, vi, ci in zip(x, values, colors):
    ax.scatter([xi], [vi], color=ci, s=120, zorder=4)

ax.axhline(ideal_obs_Z7, color="green", linestyle="--", linewidth=2,
           label=f"Ideal = {ideal_obs_Z7}", zorder=2)

ax.set_xticks(x)
ax.set_xticklabels(methods)
ax.set_ylabel(r"$\langle Z_7 \rangle$", fontsize=13)
ax.set_title(f"15-qubit Ising mirror — PEC+SLC at bias_tolerance={chosen_bias_thres}",
             fontsize=12)
ax.legend(loc="lower right")
ax.grid(axis="y", alpha=0.3)
#ax.set_ylim(0, max(1.2, max(values) * 1.2))
plt.tight_layout()
plt.show()

### 3.2.10 結論

SLC の手法の鍵となる利点は、サンプリングコストの削減です。対象のオブザーバブルの伝播したライトコーンの内側にあるノイズ生成子だけを保つことで、完全な PEC のオーバーヘッド $\gamma^2$ が、選んだ `bias_tolerance` において下がります。トレードオフはコストに対するバイアスです。わずかに許容したバイアスが、サンプリングオーバーヘッドの大きな削減を買います。

この15量子ビットのイジングミラー上の $Z_7$ のような局所オブザーバブルでは、ライトコーンが十分に狭いので、枝刈り（pruning）しても期待値が保たれます。回復された $\langle Z_7\rangle$ は理想値 1 に近く、素の PEC と比べてサンプリングオーバーヘッドが削減されています。

これで第3章が終わります。3つの構造を意識したエラー緩和の戦略を扱いました。

- PNA: 反ノイズを吸収するためにオブザーバブルを書き換える
- PEC: 反ノイズで回路を書き換える
- SLC: shaded lightcone を使って PEC のノイズ生成子を枝刈りする

これらの手法はすべて、第2章の Samplomatic + NoiseLearnerV3 の基盤の上に構築されています。

おめでとうございます！Lab 3 を完了しました。

次の Lab 4 では、量子優位性の概念と、それをどう測るかを紹介します。実験を評価するための IBM の Quantum Advantage Tracker の使い方も含みます。また、量子ビット数を減らすためのエラー緩和の手法とスマートなエンコード戦略を用いて量子最適化アルゴリズムを実装することで、これらの考えを実践的に実証します。

# 追加情報

**作成者:** Sophy Shin & Sophie Engineer

**バージョン:** 1.0.0